# Home Credit 10 — Feature Subset Search and Robustness

**目的：** 使用 Notebook 09 的正式特征排序路线，在固定模型配置和受控时间窗口中评估不同子集深度、局部成员调整及跨时间稳定性。

```text
NB09 Formal Ranking Routes
      ↓
Stage 1: Coarse Search (Top-200 ... Top-800)
      ↓
Stage 2A: Boundary Expansion
      ↓
Stage 2B: Fine Search (±50)
      ↓
Conditional Ranking-Variant Follow-up
      ↓
Stage 3: Restricted Local Repair
      ↓
Stage 4: F1–F4 Temporal Evaluation
      ↓
Pareto Review & 1-SE-style Plateau Diagnostics
      ↓
Freeze Checks / Handoff to NB11
```

搜索过程优先比较具有实际模型证据的路线与特征数量，再将昂贵的时间评估集中到少量候选。这里的 F1–F4 属于 **Train 内开发阶段时间诊断**，并非独立的最终泛化性能评估。

**数据边界：** Train Week 0–63 可用于本阶段；Validation Week 64–77 和 OOT Week 78–91 保持隔离。Tree 可以在本阶段形成特征来源集合；LR 还需在后续 Notebook 11 中完成特征表示搜索，不能把本阶段筛出的原始来源变量直接视为最终设计矩阵。


## 1. 搜索维度：Route、Depth 与局部成员调整

Notebook 10 的 Feature Selection 可以拆成三个不同的问题：

```text
Search
=
Route
×
Depth
+
Local Composition Repair
```

其中：

```text
NB9
→ Route
→ 按哪一条 feature ranking route 搜索？

NB10
→ Depth
→ 沿这条 ranking route 应该走到多少个 features？

NB10 Local Repair
→ Composition
→ 当前 Top-k 附近是否存在少量 membership error？
```

这三个问题不能混在一起。

### Route：先决定沿哪一条排序路线搜索

Notebook 9 已经根据 Notebook 8 的 feature evidence 构造了多条正式 ranking routes。

例如，不同 route 会按照不同证据给 features 排出不同优先级：

```text
Route A:
Feature A
→ Feature B
→ Feature C
→ ...

Route B:
Feature C
→ Feature A
→ Feature D
→ ...
```

因此 Notebook 9 解决的是：

> **哪些 features 应该优先进入搜索？**

但 ranking route 本身还不能回答：

```text
应该取 Top300？
Top500？
还是 Top700？
```

所以 Notebook 10 不需要重新构造新的全局 ranking，而是直接沿 Notebook 9 已经生成的 routes 继续搜索。

---

### Depth：决定一条 Route 应该走多深

假设某条 ranking route 是：

```text
Feature 1
→ Feature 2
→ Feature 3
→ ...
→ Feature 1209
```

Notebook 10 需要进一步决定：

```text
Top200
Top300
Top400
Top500
Top600
Top700
Top800
```

哪一个 depth 能得到更好的模型表现与稳定性。

因此 Notebook 10 的核心问题之一是：

> **在一条已经确定的 ranking route 上，应该在哪里截断？**

---

### Local Composition Repair：修正固定 Top-k 的局部错误

即使找到了一个较好的 depth，例如：

```text
Top550
```

也不代表前 550 个 features 的 membership 一定完全正确。

例如，可能出现：

```text
rank 420
→ 实际贡献较弱

rank 558
→ 虽然刚好落在 Top550 外，
   但加入后可能改善模型
```

如果始终严格使用 Top-k nesting：

```text
Top550
=
rank 1 到 rank 550
```

就无法发现这种局部 membership error。

因此 Notebook 10 后面还允许：

```text
少量 drop
+
少量 add
```

对 finalist 附近进行 Restricted Local Repair。

---

所以 Notebook 10 真正搜索的不是任意 feature subset，而是：

```text
Route
×
Depth
+
Finalist 附近有限的 Local Composition Repair
```

也就是说：

```text
NB9
→ 决定搜索路线

NB10
→ 决定路线走多深

NB10 Local Repair
→ 对最终候选做有限的成员修正
```

至于如何在有限计算预算下搜索这些候选，将在下面的整体 heuristic 中定义。


## 2. 数据边界与模块职责

```text
Train       Week 0–63   → 本 Notebook可以使用
Validation  Week 64–77  → 本 Notebook不读取
OOT         Week 78–91  → 本 Notebook不读取
```

F1–F4 仍然只是：

> **Development Temporal Robustness Diagnostic**

不是最终无偏性能估计。

因为 Notebook 8 的 supervised evidence 已经使用过 Week 0–63。

真正独立的 future holdout 仍然是：

```text
Validation Week 64–77
```

它必须等 feature lists 冻结以后，才由 Notebook 11 打开。

---

### Notebook 10 不再做什么？

它不重新：

- 计算 IV / MI；
- 建 redundancy graph；
- 计算 Gain / Permutation evidence；
- 建 interaction evidence；
- 重新定义 NB9 ranking routes。

原因不是这些不重要，而是：

> **它们已经在上游解决。**

如果 NB10 又重新生成 ranking：

```text
NB9 决定 Direction
```

这一层职责就会被破坏。

因此新版 NB10 应直接读取 NB9 的：

```text
tree_rankings.json
lr_rankings.json
nb9_ranking_config.json
```

然后只做真正昂贵的 subset evaluation。

这也是一种成本控制：

> **上游已经回答的问题，不在下游重复付费。**


### 执行配置与性能优化

- 启用 `RUN_LOCAL_REPAIR = True` 与 `RUN_FULL_TEMPORAL = True`，以覆盖配置的搜索及评估阶段。
- 使用 Colab 本地缓存读取 Train Parquet，减少重复的 Google Drive I/O。
- 对 `WEEK_NUM` row group 数据进行缓存，避免 Tree 路线评估阶段重复解码。
- 不改变固定模型参数、时间折定义、排序规则和最终冻结检查条件。

已有输出可能包含 checkpoint / cache 命中记录；这些标记表示本次运行复用了已保存的计算结果。


In [19]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [20]:
from pathlib import Path
from collections import defaultdict
import gc
import json
import time
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import lightgbm as lgb

from scipy import sparse
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from IPython.display import display


# ============================================================
# Basic constants
# ============================================================

ID_COL = "case_id"
TARGET_COL = "target"
WEEK_COL = "WEEK_NUM"
RANDOM_SEED = 42


# ============================================================
# Search execution budget
# ============================================================

COARSE_SIZES = [
    200,
    300,
    400,
    500,
    600,
    700,
    800,
]

FINE_STEP = 50

MIN_SEARCH_SIZE = 100
MAX_SEARCH_SIZE = 1000

BOUNDARY_EXPANSION_STEP = 100
MAX_BOUNDARY_ROUNDS = 2

LOCAL_REPAIR_PARENT_BUDGET = 4
LOCAL_REPAIR_DROP_POOL = 8
LOCAL_REPAIR_ADD_POOL = 12
LOCAL_REPAIR_PROPOSALS_PER_PARENT = 12
MAX_LOCAL_REPAIR_ROUNDS = 2

MAX_FULL_TEMPORAL_PER_BRANCH = 4


# ============================================================
# Execution switches
# ============================================================

RUN_TREE_COARSE = True
RUN_LR_COARSE = True

RUN_FINE_SEARCH = True
RUN_BOUNDARY_EXPANSION = True

RUN_RELEVANCE_VARIANT_FOLLOWUP = True

RELEVANCE_VARIANTS_TO_TEST = [
    "iv60_mi40",
    "iv40_mi60",
]

RUN_LOCAL_REPAIR = False
RUN_FULL_TEMPORAL = True


# ============================================================
# Baselines
# ============================================================

DEVICE_TYPE = "gpu"

# Performance only: cache the unchanged Train parquet on Colab local disk.
# This avoids repeatedly reading the same large file through Google Drive.
USE_LOCAL_TRAIN_CACHE = True

LGBM_MAX_BIN = 63
LGBM_ENABLE_BUNDLE = True
LGBM_MAX_CATEGORIES = 100

LR_EPOCHS = 3
LR_ALPHA = 1e-4
LR_ETA0 = 0.01


# ============================================================
# Google Drive
# ============================================================

try:
    from google.colab import drive

except ImportError as exc:

    raise RuntimeError(
        "Notebook 10 当前版本按 Google Colab + "
        "Google Drive 配置，请在 Colab 中运行。"
    ) from exc


drive.mount(
    "/content/drive"
)


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/HomeCredit_Project"
)

ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "artifacts"
)

REPORT_ROOT = (
    ARTIFACT_ROOT
    / "nb10"
    / "final"
)

CHECKPOINT_ROOT = (
    ARTIFACT_ROOT
    / "nb10"
    / "checkpoints"
)

AUTO_EXTRACT_ROOT = (
    ARTIFACT_ROOT
    / "_auto_extracted"
)


REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUTO_EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"Project root 不存在：{PROJECT_ROOT}"
    )


# ============================================================
# Unified upstream helpers
# ============================================================

def _find_complete_dir(
    root,
    required_files,
):
    """
    在 root 及其子目录中寻找：
    required_files 全部位于同一个目录的情况。
    """

    root = Path(
        root
    )

    if not root.exists():
        return None

    required_files = set(
        required_files
    )


    # --------------------------------------------------------
    # First check root itself
    # --------------------------------------------------------

    if root.is_dir():

        try:

            names = {
                path.name
                for path in root.iterdir()
            }

        except OSError:

            names = set()


        if required_files.issubset(
            names
        ):

            return root


    # --------------------------------------------------------
    # Recursive search
    # --------------------------------------------------------

    first_required = sorted(
        required_files
    )[0]


    for first_file in root.rglob(
        first_required
    ):

        parent = (
            first_file.parent
        )

        try:

            names = {
                path.name
                for path in parent.iterdir()
            }

        except OSError:

            continue


        if required_files.issubset(
            names
        ):

            return parent


    return None



def _zip_contains_required_files(
    zip_path,
    required_files,
):
    """
    检查 ZIP 是否至少包含这些文件名。
    """

    required_files = set(
        required_files
    )


    try:

        with zipfile.ZipFile(
            zip_path,
            "r",
        ) as zip_file:

            basenames = {
                Path(name).name
                for name in zip_file.namelist()
                if not name.endswith("/")
            }


    except zipfile.BadZipFile:

        return False


    return required_files.issubset(
        basenames
    )



def _extract_zip(
    zip_path,
    label,
):
    """
    把 ZIP 自动解压到统一缓存目录。

    原始 ZIP 不修改。
    """

    zip_path = Path(
        zip_path
    )


    destination = (
        AUTO_EXTRACT_ROOT
        / label.replace(
            " ",
            "_"
        )
        / zip_path.stem
    )


    destination.mkdir(
        parents=True,
        exist_ok=True,
    )


    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:

        zip_file.extractall(
            destination
        )


    return destination



def find_required_dir(
    required_files,
    preferred_candidates,
    label,
):
    """
    Unified upstream resolution:

    1. 先寻找已经解压好的文件；
    2. 如果只有 ZIP，则自动识别并解压；
    3. 如果约定目录没有，再在当前
       HomeCredit_Project 内做 fallback search；
    4. 不扫描旧 Kaggle 路径。
    """

    required_files = set(
        required_files
    )


    # ========================================================
    # 1. Preferred directories
    # ========================================================

    for candidate in preferred_candidates:

        candidate = Path(
            candidate
        )


        resolved = _find_complete_dir(
            candidate,
            required_files,
        )


        if resolved is not None:

            print(
                f"{label}: using extracted files -> "
                f"{resolved}"
            )

            return resolved


    # ========================================================
    # 2. ZIP files inside preferred locations
    # ========================================================

    checked_zips = set()


    for candidate in preferred_candidates:

        candidate = Path(
            candidate
        )


        if candidate.exists():

            search_root = (
                candidate
                if candidate.is_dir()
                else candidate.parent
            )

        else:

            search_root = (
                candidate.parent
            )


        if not search_root.exists():

            continue


        for zip_path in search_root.rglob(
            "*.zip"
        ):

            zip_path = (
                zip_path.resolve()
            )


            if zip_path in checked_zips:

                continue


            checked_zips.add(
                zip_path
            )


            if not _zip_contains_required_files(
                zip_path,
                required_files,
            ):

                continue


            extracted_root = _extract_zip(
                zip_path,
                label,
            )


            resolved = _find_complete_dir(
                extracted_root,
                required_files,
            )


            if resolved is not None:

                print(
                    f"{label}: ZIP detected -> "
                    f"{zip_path.name}"
                )

                print(
                    f"{label}: automatically extracted -> "
                    f"{resolved}"
                )

                return resolved


    # ========================================================
    # 3. Project-level extracted fallback
    # ========================================================

    resolved = _find_complete_dir(
        PROJECT_ROOT,
        required_files,
    )


    if resolved is not None:

        print(
            f"{label}: project fallback -> "
            f"{resolved}"
        )

        return resolved


    # ========================================================
    # 4. Project-level ZIP fallback
    # ========================================================

    for zip_path in PROJECT_ROOT.rglob(
        "*.zip"
    ):

        zip_path = (
            zip_path.resolve()
        )


        if zip_path in checked_zips:

            continue


        checked_zips.add(
            zip_path
        )


        if not _zip_contains_required_files(
            zip_path,
            required_files,
        ):

            continue


        extracted_root = _extract_zip(
            zip_path,
            label,
        )


        resolved = _find_complete_dir(
            extracted_root,
            required_files,
        )


        if resolved is not None:

            print(
                f"{label}: fallback ZIP detected -> "
                f"{zip_path.name}"
            )

            print(
                f"{label}: automatically extracted -> "
                f"{resolved}"
            )

            return resolved


    raise FileNotFoundError(
        f"找不到 {label} 的完整上游输出：\n"
        f"{sorted(required_files)}\n\n"
        "已经检查：\n"
        "- preferred directories\n"
        "- preferred ZIP files\n"
        "- HomeCredit_Project extracted files\n"
        "- HomeCredit_Project ZIP files\n\n"
        f"Project root: {PROJECT_ROOT}"
    )



def assert_required_files(
    directory,
    required_files,
    label,
):

    directory = Path(
        directory
    )


    missing = [
        filename
        for filename in required_files
        if not (
            directory
            / filename
        ).exists()
    ]


    if missing:

        raise FileNotFoundError(
            f"{label} 缺少文件：{missing}\n"
            f"当前目录：{directory}"
        )


# ============================================================
# Notebook 8
#
# NB8 后面的代码本来就是直接从 ZIP 读取，
# 所以这里保留 ZIP handoff。
#
# 但是不再死认一个 ZIP 文件名：
# 只要 nb08 目录下有一个 ZIP，
# 且内部包含正确文件，就自动使用。
# ============================================================

NB8_REQUIRED_MEMBERS = {
    (
        "reports/feature_evidence/"
        "feature_evidence_report.csv"
    ),
    (
        "reports/feature_evidence/"
        "nb8_feature_evidence_config.json"
    ),
}



def resolve_nb8_zip():

    nb8_root = (
        ARTIFACT_ROOT
        / "nb08"
    )


    preferred = (
        nb8_root
        / "nb08_feature_evidence_outputs.zip"
    )


    candidates = []


    if preferred.exists():

        candidates.append(
            preferred
        )


    if nb8_root.exists():

        candidates.extend(
            sorted(
                nb8_root.rglob(
                    "*.zip"
                )
            )
        )


    checked = set()


    for zip_path in candidates:

        zip_path = Path(
            zip_path
        ).resolve()


        if zip_path in checked:

            continue


        checked.add(
            zip_path
        )


        try:

            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zip_file:

                members = set(
                    zip_file.namelist()
                )


                if NB8_REQUIRED_MEMBERS.issubset(
                    members
                ):

                    return zip_path


        except zipfile.BadZipFile:

            continue


    raise FileNotFoundError(
        "找不到可用的 Notebook 8 evidence ZIP。\n"
        f"已检查：{nb8_root}\n\n"
        "ZIP 必须包含：\n"
        + "\n".join(
            sorted(
                NB8_REQUIRED_MEMBERS
            )
        )
    )



NB8_ZIP = resolve_nb8_zip()


# ============================================================
# Notebook 9
#
# NB9 后面读取普通 JSON / CSV。
# 如果只有 ZIP，自动解压。
# ============================================================

NB9_OUTPUT_DIR = find_required_dir(
    required_files={
        "tree_rankings.json",
        "lr_rankings.json",
        "nb9_ranking_config.json",
    },

    preferred_candidates=[
        (
            ARTIFACT_ROOT
            / "nb09"
            / "final"
        ),
        (
            ARTIFACT_ROOT
            / "nb09"
        ),
    ],

    label="Notebook 9",
)


# ============================================================
# Status
# ============================================================

print(
    "\n=============================="
)

print(
    "Project:",
    PROJECT_ROOT,
)

print(
    "Notebook 8 ZIP:",
    NB8_ZIP,
)

print(
    "Notebook 9 resolved:",
    NB9_OUTPUT_DIR,
)

print(
    "Notebook 10 checkpoint:",
    CHECKPOINT_ROOT,
)

print(
    "Notebook 10 final:",
    REPORT_ROOT,
)

print(
    "==============================\n"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook 9: using extracted files -> /content/drive/MyDrive/HomeCredit_Project/artifacts/nb09/final

Project: /content/drive/MyDrive/HomeCredit_Project
Notebook 8 ZIP: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb08/nb08_feature_evidence_outputs.zip
Notebook 9 resolved: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb09/final
Notebook 10 checkpoint: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/checkpoints
Notebook 10 final: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/final



## 3. 上游排序结果与环境依赖

本 Notebook需要四类上游对象：

```text
Notebook 9
→ Tree formal ranking paths
→ LR formal ranking paths
→ ranking config / promoted sensitivity variants

Notebook 8
→ reference-model configuration
→ 后续 Local Repair 需要时使用 atomic evidence

Notebook 7
→ LR preprocessing rules

Notebook 5
→ train_clean.parquet
```


### Colab 路径约定

Notebook 9 的实际运行结果已经确认：

```text
/content/drive/MyDrive/HomeCredit_Project
└── artifacts
    ├── nb08
    │   └── nb08_feature_evidence_outputs.zip
    ├── nb09
    │   └── final
    └── nb10
        ├── checkpoints
        └── final
```

因此本 Notebook不再把 `/kaggle/input` 或 `/kaggle/working` 当作 fallback。

NB8 继续按 Notebook 9 的方式 **直接从 ZIP 读取 evidence**；
NB9 直接读取 `artifacts/nb09/final`。
旧 NB5 / NB7 若当时保存目录名称不同，只在当前
`HomeCredit_Project` 内做一次 fallback search。

最关键的变化是：

> **NB10 不再复制 NB9 的 ranking-construction code。**

否则会出现一个很危险的问题：

```text
NB9 已经因为 sensitivity calibration 改了 route
        ↓
NB10 仍然按旧逻辑自己重建 route
        ↓
表面上“接力”了 NB9
实际上搜索的是另一套空间
```

所以 Direction 必须只有一个来源：

```text
NB9 final outputs
```

NB10 只负责执行。

### 为什么仍然需要 NB8？

不是为了重新排序，而是因为：

- Tree baseline 参数可沿用 NB8 reference configuration；
- Restricted Local Repair 后面可能需要读取 atomic evidence，构造少量有理由的 add/drop candidates。

但这些都不能把 NB10 重新变成一个新的 ranking notebook。


In [21]:
# ============================================================
# 1. Upstream handoff
# ============================================================

# ------------------------------------------------------------
# Notebook 9 — formal ranking paths
# ------------------------------------------------------------

assert_required_files(
    NB9_OUTPUT_DIR,
    {
        "tree_rankings.json",
        "lr_rankings.json",
        "nb9_ranking_config.json",
    },
    "Notebook 9",
)


# ------------------------------------------------------------
# Notebook 8 — ZIP evidence handoff
# ------------------------------------------------------------
#
# NB9 的真实 Colab 流程没有把 NB8 evidence 解压成
# artifacts/nb08/final，而是直接读取：
#
# artifacts/nb08/nb08_feature_evidence_outputs.zip
#
# NB10 保持同一接口，避免要求重复解压。
NB8_MEMBERS = {
    "feature_evidence": (
        "reports/feature_evidence/"
        "feature_evidence_report.csv"
    ),
    "config": (
        "reports/feature_evidence/"
        "nb8_feature_evidence_config.json"
    ),
}


def read_csv_from_zip(
    zip_path,
    internal_path,
):
    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:
        with zip_file.open(
            internal_path
        ) as file:
            return pd.read_csv(
                file
            )


def read_json_from_zip(
    zip_path,
    internal_path,
):
    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:
        with zip_file.open(
            internal_path
        ) as file:
            return json.load(
                file
            )


# ------------------------------------------------------------
# Notebook 7 — LR preprocessing
# ------------------------------------------------------------

NB7_OUTPUT_DIR = find_required_dir(
    required_files={
        "numeric_preprocessing_stats.csv",
        "categorical_encoding_rules.json",
        "nb7_candidate_features.csv",
    },
    preferred_candidates=[
        (
            ARTIFACT_ROOT
            / "nb07"
            / "final"
        ),
        (
            ARTIFACT_ROOT
            / "nb07"
        ),
        (
            PROJECT_ROOT
            / "reports"
            / "reference_model"
        ),
    ],
    label="Notebook 7",
)


# ------------------------------------------------------------
# Notebook 5 — clean Train
# ------------------------------------------------------------

NB5_OUTPUT_DIR = find_required_dir(
    required_files={
        "train_clean.parquet",
        "feature_catalog.csv",
    },
    preferred_candidates=[
        (
            ARTIFACT_ROOT
            / "nb05"
            / "final"
        ),
        (
            ARTIFACT_ROOT
            / "nb05"
        ),
        (
            PROJECT_ROOT
            / "clean_split"
        ),
    ],
    label="Notebook 5",
)


print(
    "Resolved Notebook 7:",
    NB7_OUTPUT_DIR,
)

print(
    "Resolved Notebook 5:",
    NB5_OUTPUT_DIR,
)


# ------------------------------------------------------------
# Read Notebook 9 routes
# ------------------------------------------------------------

with open(
    NB9_OUTPUT_DIR / "tree_rankings.json",
    "r",
    encoding="utf-8",
) as file:
    tree_routes = json.load(file)

with open(
    NB9_OUTPUT_DIR / "lr_rankings.json",
    "r",
    encoding="utf-8",
) as file:
    lr_routes = json.load(file)

with open(
    NB9_OUTPUT_DIR / "nb9_ranking_config.json",
    "r",
    encoding="utf-8",
) as file:
    nb9_config = json.load(file)


# ------------------------------------------------------------
# Optional Notebook 9 diagnostics
# ------------------------------------------------------------
# 这些表只用于 preflight review。
# NB10 不根据它们自动删除 route；formal routes 仍以
# tree_rankings.json / lr_rankings.json 为准。
NB9_DIAGNOSTIC_FILES = [
    "tree_route_overlap.csv",
    "lr_route_overlap.csv",
    "relevance_ranking_sensitivity.csv",
    "relevance_membership_changes_vs_50_50.csv",
    "gain_permutation_agreement.csv",
    "interaction_pair_limit_sensitivity.csv",
    "interaction_rescue_recalibration.csv",
    "temporal_boundary_diagnostic.csv",
]

nb9_diagnostics = {}

for filename in NB9_DIAGNOSTIC_FILES:
    path = NB9_OUTPUT_DIR / filename
    if path.exists():
        nb9_diagnostics[filename] = pd.read_csv(path)

print(
    "NB9 diagnostics found:",
    sorted(nb9_diagnostics),
)


# ------------------------------------------------------------
# Optional NB9 ranking variants already constructed upstream
# ------------------------------------------------------------
# NB10 不自己重算这些 variants。
# 如果 NB9 已经保存，则只在后面按预算条件评估。
RELEVANCE_SENSITIVITY_PATH = (
    NB9_OUTPUT_DIR
    / "relevance_sensitivity_rankings.json"
)

relevance_sensitivity_orders = {}

if RELEVANCE_SENSITIVITY_PATH.exists():
    with open(
        RELEVANCE_SENSITIVITY_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        relevance_sensitivity_orders = json.load(file)


# ------------------------------------------------------------
# NB9 → NB10 operational guardrails
# ------------------------------------------------------------
# 这些阈值不是统计显著性检验。
# 它们只是帮助 NB10 决定“哪些不确定性值得继续付搜索成本”。
def _diag(name):
    return nb9_diagnostics.get(
        name,
        pd.DataFrame(),
    )


relevance_change = _diag(
    "relevance_membership_changes_vs_50_50.csv"
)

if not relevance_change.empty:
    local_relevance = relevance_change.loc[
        relevance_change["variant"].isin(
            ["iv60_mi40", "iv40_mi60"]
        )
    ].copy()

    if not local_relevance.empty:
        local_relevance[
            "changed_fraction"
        ] = (
            local_relevance[
                "entered_count"
            ]
            / local_relevance[
                "k"
            ]
        )

        relevance_local_max_changed_fraction = float(
            local_relevance[
                "changed_fraction"
            ].max()
        )
    else:
        relevance_local_max_changed_fraction = np.nan
else:
    relevance_local_max_changed_fraction = np.nan


interaction_sensitivity = _diag(
    "interaction_pair_limit_sensitivity.csv"
)

if not interaction_sensitivity.empty:
    interaction_last = (
        interaction_sensitivity
        .sort_values("pair_limit")
        .iloc[-1]
    )
    interaction_largest_limit_jaccard = float(
        interaction_last[
            "jaccard_vs_formal_limit"
        ]
    )
else:
    interaction_largest_limit_jaccard = np.nan


temporal_boundary = _diag(
    "temporal_boundary_diagnostic.csv"
)

if not temporal_boundary.empty:
    temporal_near_zero_fraction = float(
        temporal_boundary.iloc[0][
            "near_zero_fraction"
        ]
    )
else:
    temporal_near_zero_fraction = np.nan


gain_perm = _diag(
    "gain_permutation_agreement.csv"
)

if not gain_perm.empty:
    gain_permutation_spearman = float(
        gain_perm.iloc[0][
            "spearman_rank_corr"
        ]
    )
else:
    gain_permutation_spearman = np.nan


NB9_GUARDRAILS = {
    "relevance_local_variant_material": (
        pd.notna(
            relevance_local_max_changed_fraction
        )
        and relevance_local_max_changed_fraction
        >= 0.10
    ),
    "interaction_pair_limit_sensitive": (
        pd.notna(
            interaction_largest_limit_jaccard
        )
        and interaction_largest_limit_jaccard
        < 0.80
    ),
    "temporal_boundary_sensitive": (
        pd.notna(
            temporal_near_zero_fraction
        )
        and temporal_near_zero_fraction
        >= 0.20
    ),
    "gain_permutation_disagreement": (
        pd.notna(
            gain_permutation_spearman
        )
        and gain_permutation_spearman
        < 0.75
    ),
}


nb9_preflight = pd.DataFrame([
    {
        "check": "Relevance nearby weighting",
        "observed_value": (
            relevance_local_max_changed_fraction
        ),
        "guardrail_triggered": (
            NB9_GUARDRAILS[
                "relevance_local_variant_material"
            ]
        ),
        "NB10_action": (
            "If base Relevance survives, test only nearby "
            "NB9-saved variants at the same competitive depth."
        ),
    },
    {
        "check": "Interaction pair-limit",
        "observed_value": (
            interaction_largest_limit_jaccard
        ),
        "guardrail_triggered": (
            NB9_GUARDRAILS[
                "interaction_pair_limit_sensitive"
            ]
        ),
        "NB10_action": (
            "Run base route now; if it reaches the final set, "
            "require an NB9 route-definition follow-up before freeze."
        ),
    },
    {
        "check": "Temporal zero-boundary",
        "observed_value": (
            temporal_near_zero_fraction
        ),
        "guardrail_triggered": (
            NB9_GUARDRAILS[
                "temporal_boundary_sensitive"
            ]
        ),
        "NB10_action": (
            "Run base route now; if it reaches the final set, "
            "require a threshold/continuous-ranking follow-up before freeze."
        ),
    },
    {
        "check": "Gain vs Permutation",
        "observed_value": (
            gain_permutation_spearman
        ),
        "guardrail_triggered": (
            NB9_GUARDRAILS[
                "gain_permutation_disagreement"
            ]
        ),
        "NB10_action": (
            "Do not invent a partial Permutation route; keep it "
            "as a diagnostic / local-repair clue."
        ),
    },
])

print(
    "NB9 → NB10 preflight guardrails"
)
display(
    nb9_preflight
)


# ------------------------------------------------------------
# Read Notebook 8 evidence/config directly from ZIP
# ------------------------------------------------------------

feature_evidence = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS[
        "feature_evidence"
    ],
)

nb8_config = read_json_from_zip(
    NB8_ZIP,
    NB8_MEMBERS[
        "config"
    ],
)


# 保持 NB10 fixed Tree baseline 与 NB8 reference setting 尽量一致。
# 旧 NB8 config 若没有记录 max_bin，则回退到本项目已运行的 255。
LGBM_MAX_BIN = int(
    nb8_config.get(
        "lgbm_max_bin",
        LGBM_MAX_BIN,
    )
)
# Force the same setting as Notebook 7
LGBM_MAX_BIN = 63

# ------------------------------------------------------------
# Read Notebook 7 preprocessing
# ------------------------------------------------------------

numeric_stats = pd.read_csv(
    NB7_OUTPUT_DIR
    / "numeric_preprocessing_stats.csv"
)

nb7_catalog = pd.read_csv(
    NB7_OUTPUT_DIR
    / "nb7_candidate_features.csv"
)

with open(
    NB7_OUTPUT_DIR
    / "categorical_encoding_rules.json",
    "r",
    encoding="utf-8",
) as file:
    lr_category_lists = json.load(file)


# ------------------------------------------------------------
# Read clean Train
# ------------------------------------------------------------

DRIVE_TRAIN_PATH = (
    NB5_OUTPUT_DIR
    / "train_clean.parquet"
)

if USE_LOCAL_TRAIN_CACHE:
    LOCAL_TRAIN_CACHE_ROOT = Path(
        "/content/nb10_local_cache"
    )
    LOCAL_TRAIN_CACHE_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    LOCAL_TRAIN_PATH = (
        LOCAL_TRAIN_CACHE_ROOT
        / DRIVE_TRAIN_PATH.name
    )

    source_size = (
        DRIVE_TRAIN_PATH
        .stat()
        .st_size
    )

    local_cache_valid = (
        LOCAL_TRAIN_PATH.exists()
        and LOCAL_TRAIN_PATH.stat().st_size
        == source_size
    )

    if not local_cache_valid:
        print(
            "Caching train_clean.parquet from Google Drive "
            "to Colab local disk..."
        )
        print(
            f"  size = {source_size / (1024 ** 3):.2f} GiB"
        )
        shutil.copy2(
            DRIVE_TRAIN_PATH,
            LOCAL_TRAIN_PATH,
        )

    TRAIN_PATH = LOCAL_TRAIN_PATH
    print(
        "Train parquet source: Colab local cache ->",
        TRAIN_PATH,
    )

else:
    TRAIN_PATH = DRIVE_TRAIN_PATH
    print(
        "Train parquet source: Google Drive ->",
        TRAIN_PATH,
    )

train_parquet = pq.ParquetFile(
    TRAIN_PATH
)

train_meta = pd.read_parquet(
    TRAIN_PATH,
    columns=[
        ID_COL,
        WEEK_COL,
        TARGET_COL,
    ],
    engine="pyarrow",
).reset_index(
    drop=True
)

assert train_meta[
    ID_COL
].is_unique

assert train_meta[
    WEEK_COL
].between(
    0,
    63,
).all()


schema_columns = set(
    train_parquet.schema.names
)

candidate_features = (
    feature_evidence
    .loc[
        feature_evidence[
            "feature"
        ].isin(
            schema_columns
        ),
        "feature",
    ]
    .drop_duplicates()
    .tolist()
)

feature_evidence = (
    feature_evidence
    .loc[
        feature_evidence[
            "feature"
        ].isin(
            candidate_features
        )
    ]
    .drop_duplicates(
        subset=["feature"]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Validate ranking handoff
# ------------------------------------------------------------

def validate_route_dict(
    routes,
    branch,
):
    candidate_set = set(
        candidate_features
    )

    for route_name, order in (
        routes.items()
    ):

        if len(order) != len(
            set(order)
        ):
            raise ValueError(
                f"{branch}/{route_name}: "
                "duplicate features in ranking"
            )

        unknown = (
            set(order)
            - candidate_set
        )

        if unknown:
            raise ValueError(
                f"{branch}/{route_name}: "
                f"unknown features: "
                f"{sorted(unknown)[:10]}"
            )

        if len(order) != len(
            candidate_features
        ):
            raise ValueError(
                f"{branch}/{route_name}: "
                "ranking does not cover the "
                "complete candidate pool"
            )


validate_route_dict(
    tree_routes,
    "tree",
)

validate_route_dict(
    lr_routes,
    "lr",
)


# Fixed Tree baseline inherits NB8 reference parameters.
LGBM_NUM_BOOST_ROUND = int(
    nb8_config.get(
        "lgbm_n_estimators",
        300,
    )
)

LGBM_BIN_SAMPLE_COUNT = int(
    nb8_config.get(
        "lgbm_bin_construct_sample_cnt",
        50_000,
    )
)


print(
    "Candidate features:",
    len(candidate_features),
)

print(
    "Tree routes:",
    list(tree_routes),
)

print(
    "LR routes:",
    list(lr_routes),
)

print(
    "NB9 promoted Tree variants:",
    nb9_config.get(
        "promoted_tree_relevance_variants",
        [],
    ),
)

print(
    "NB9 promoted LR variants:",
    nb9_config.get(
        "promoted_lr_relevance_variants",
        [],
    ),
)

print(
    "Validation / OOT read:",
    False,
)


print(
    "NB10 checkpoint root:",
    CHECKPOINT_ROOT,
)

for filename, table in nb9_diagnostics.items():
    print("\nNB9 diagnostic:", filename)
    display(table.head(20))


Notebook 7: using extracted files -> /content/drive/MyDrive/HomeCredit_Project/artifacts/nb07/final
Notebook 5: ZIP detected -> nb05_clean_split_outputs.zip
Notebook 5: automatically extracted -> /content/drive/MyDrive/HomeCredit_Project/artifacts/_auto_extracted/Notebook_5/nb05_clean_split_outputs/clean_split
Resolved Notebook 7: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb07/final
Resolved Notebook 5: /content/drive/MyDrive/HomeCredit_Project/artifacts/_auto_extracted/Notebook_5/nb05_clean_split_outputs/clean_split
NB9 diagnostics found: ['gain_permutation_agreement.csv', 'interaction_pair_limit_sensitivity.csv', 'interaction_rescue_recalibration.csv', 'lr_route_overlap.csv', 'relevance_membership_changes_vs_50_50.csv', 'relevance_ranking_sensitivity.csv', 'temporal_boundary_diagnostic.csv', 'tree_route_overlap.csv']
NB9 → NB10 preflight guardrails


,check,observed_value,guardrail_triggered,NB10_action
0,Relevance nearby weighting,0.235000,True,"If base Relevance survives, test only nearby N..."
1,Interaction pair-limit,0.535211,True,Run base route now; if it reaches the final se...
2,Temporal zero-boundary,0.452440,True,Run base route now; if it reaches the final se...
3,Gain vs Permutation,0.545980,True,Do not invent a partial Permutation route; kee...


Train parquet source: Colab local cache -> /content/nb10_local_cache/train_clean.parquet
Candidate features: 1209
Tree routes: ['relevance', 'model_gain', 'temporal_relevance', 'interaction_relevance', 'redundancy_aware', 'consensus']
LR routes: ['relevance', 'temporal_relevance', 'redundancy_aware', 'lr_consensus']
NB9 promoted Tree variants: []
NB9 promoted LR variants: []
Validation / OOT read: False
NB10 checkpoint root: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/checkpoints

NB9 diagnostic: tree_route_overlap.csv


,route_a,route_b,k,overlap_count,jaccard
0,relevance,model_gain,200,94,0.307190
1,relevance,temporal_relevance,200,113,0.393728
2,relevance,interaction_relevance,200,162,0.680672
3,relevance,redundancy_aware,200,104,0.351351
4,relevance,consensus,200,142,0.550388
5,model_gain,temporal_relevance,200,93,0.302932
6,model_gain,interaction_relevance,200,89,0.286174
7,model_gain,redundancy_aware,200,73,0.223242
8,model_gain,consensus,200,97,0.320132
9,temporal_relevance,interaction_relevance,200,95,0.311475



NB9 diagnostic: lr_route_overlap.csv


...

,pair_limit,exclusive_rescued_feature_count,new_vs_formal_limit,lost_vs_formal_limit,jaccard_vs_formal_limit,new_feature_examples
0,300,38,0,0,1.000000,NaN
1,500,52,14,0,0.730769,childnum_21L_mean|clientscnt3m_3712950L|client...
2,800,71,33,0,0.535211,childnum_21L_mean|childnum_21L_std|clientscnt3...



NB9 diagnostic: interaction_rescue_recalibration.csv


,nb8_rescue_count,exclusive_rescue_count,old_but_not_exclusive,exclusive_but_not_old
0,44,38,18,12



NB9 diagnostic: temporal_boundary_diagnostic.csv


,boundary_band,non_missing_count,near_zero_count,near_zero_fraction,negative_within_band,nonnegative_within_band
0,0.01,1209,547,0.45244,137,410


## 4. 搜索预算与候选路线

这一节是 Notebook 9 → Notebook 10 最关键的接力。

Notebook 9 已经解决：

```text
Direction
→ 同一批 1,209 个 features
→ 用不同 Ranking Algorithms
→ 形成少数 search directions
```

Notebook 10 接下来解决：

```text
Depth
→ 每条方向走到 Top200 / 300 / ... / 800 的哪里？

以及：
固定 ranking 在最有竞争力区域附近
有没有少量 membership error？
```

所以 NB10 不重新发明 Route。

但 NB9 的 **Robustness diagnostics** 会决定：

> **哪些 Route 可以直接进入 Coarse Search，哪些不确定性只需要后面做 targeted follow-up。**

---

### 4.1 Formal Routes：先全部保留进入 Coarse Search

NB9 的 Tree Top200 overlap 并不高到可以把某两条 Route 当成重复路线。

例如实际输出里：

```text
Relevance vs Interaction
Jaccard ≈ 0.681

Relevance vs Temporal
Jaccard ≈ 0.394
```

LR 的差异更明显，例如：

```text
Relevance vs Redundancy-aware
Jaccard ≈ 0.351

Temporal vs Redundancy-aware
Jaccard ≈ 0.235
```

所以现阶段没有理由在真正训练 subset model 以前，把这些 base directions 提前删掉。

因此 Stage 1 仍然使用 NB9 正式输出的：

```text
Tree:
Relevance
Model Gain
Temporal
Interaction
Redundancy-aware
Consensus

LR:
Relevance
Temporal
Redundancy-aware
LR Consensus
```

这就是：

> **NB9 负责把无穷大的空间压成少数方向；NB10 再用真实 learner performance 决定哪些方向有用。**

---

### 4.2 Relevance：有差异，但不值得一开始把搜索空间乘三

本次相对 50/50：

```text
60/40, Top200
→ 22 个 feature 进入 / 22 个退出

40/60, Top200
→ 47 个 feature 进入 / 47 个退出
```

说明 weighting 不是完全无关。

但正确反应不是：

```text
50/50 × 7 个 depth
60/40 × 7 个 depth
40/60 × 7 个 depth
...
```

全部先跑。

更合理的是：

```text
先跑 base Relevance
        ↓
如果它根本不在 Pareto front
→ 不再花钱

如果它有竞争力
→ 在同一个 competitive depth
   补 60/40、40/60
```

所以本 Notebook新增一个很小的：

> **NB9 Relevance Variant Follow-up**

它只使用 NB9 已经保存的 variant rankings，不重新构建 ranking。

---

### 4.3 Gain / Permutation：差异是真实的，但不造残缺 Route

本次共同覆盖只有：

```text
150 / 1209
≈ 12.4%
```

Gain / Permutation 的 Spearman 约：

```text
0.546
```

说明两者确实不是同一件事。

但 Permutation 没有完整覆盖 1,209 个 features。

所以 NB10 不做：

```text
Permutation Route
```

而是把它留作：

> **Local Repair 的附加 clue。**

这就是一个典型的成本/信息平衡：

```text
Evidence 有价值
≠
一定要升级成完整 Search Direction
```

---

### 4.4 Interaction：可以先搜，但如果最终赢了，Freeze 前必须复核定义敏感性

本次 exclusive rescue：

```text
300 pairs → 38 features
500 pairs → 52
800 pairs → 71
```

300 vs 800：

```text
Jaccard ≈ 0.535
```

说明当前正式 Interaction Route 对 pair cap 有明显敏感性。

但这并不意味着 NB10 现在必须停止。

更有效率的做法是：

```text
先让当前 formal Interaction Route
参加 Coarse Search
        ↓
如果它很差
→ 不再为它付更多成本

如果它进入最终竞争区
→ Freeze 前必须回 NB9
   再导出更宽 pair-limit 的 route
   做 targeted comparison
```

这就是 **conditional cost**。

---

### 4.5 Temporal：同样先搜，但赢家不能绕过 boundary guardrail

本次：

```text
547 / 1209
≈ 45.2%
```

features 的 `temporal_gini_min` 落在：

```text
[-0.01, +0.01]
```

说明 `0` 附近非常拥挤。

所以当前 label-first Temporal Route 可以继续作为一个 search direction。

但如果它最后赢：

> **必须在 Freeze 前检查 threshold/continuous ranking robustness。**

---

### 4.6 Tree 与 LR 为什么仍然分开？

Direction 虽然来自 NB9，但 subset value 是 learner-dependent。

---

因此 NB10 的第一步非常明确：

```text
NB9 Base Directions
        ↓
Tree / LR 分支分别做 wrapper evaluation
        ↓
先搜 Depth
        ↓
再根据真实表现决定
哪些 uncertainty 值得继续付钱
```


In [22]:
# ============================================================
# 2. Atomic evidence table for diagnostics / local repair only
# ============================================================

selection_table = (
    feature_evidence
    .copy()
)

for column in [
    "missing_rate",
    "iv_percentile",
    "mi_percentile",
    "relevance_score",
    "gain_percentile",
    "permutation_percentile",
    "max_shared_path_count",
    "temporal_gini_min",
    "temporal_gini_slope",
]:
    if column in selection_table.columns:
        selection_table[
            column
        ] = pd.to_numeric(
            selection_table[
                column
            ],
            errors="coerce",
        )

if (
    "interaction_rescue_candidate"
    in selection_table.columns
):
    selection_table[
        "interaction_rescue_candidate"
    ] = (
        selection_table[
            "interaction_rescue_candidate"
        ]
        .fillna(False)
        .astype(bool)
    )

if (
    "temporal_label"
    in selection_table.columns
):
    selection_table[
        "temporal_stable_flag"
    ] = (
        selection_table[
            "temporal_label"
        ]
        .eq(
            "STABLE_CANDIDATE"
        )
    )
else:
    selection_table[
        "temporal_stable_flag"
    ] = False


# Rank lookup is useful later when Local Repair asks:
# "which outside feature is just beyond this route's current boundary?"
route_rank_lookup = {
    "tree": {
        route_name: {
            feature: rank
            for rank, feature
            in enumerate(
                order,
                start=1,
            )
        }
        for route_name, order
        in tree_routes.items()
    },
    "lr": {
        route_name: {
            feature: rank
            for rank, feature
            in enumerate(
                order,
                start=1,
            )
        }
        for route_name, order
        in lr_routes.items()
    },
}


print(
    "Atomic evidence available for "
    "diagnostics/local repair:",
    selection_table.shape,
)


Atomic evidence available for diagnostics/local repair: (1209, 39)


## 5. Stage 1 — Coarse Depth Search

第一轮统一测试：

```text
200 / 300 / 400 / 500 / 600 / 700 / 800
```

如果正式 routes 数量不变，那么大致是：

```text
Tree routes × 7 sizes
LR routes   × 7 sizes
```

每个 candidate 是一个真实 subset：

```text
route A, Top200
route A, Top300
...
route B, Top200
...
```

### 为什么这笔成本必须付？

它解决的是：

> **search resolution 太低造成的 under-search。**

仅使用：

```text
300 / 500 / 800
```

中间区域几乎没有观测。

所以 500 看起来最好，可能只是因为：

```text
真正的 plateau 在 550–600
但该范围尚未经过实际评估
```

### 为什么又不直接每 10 个 features 搜一次？

因为：

```text
200, 210, 220, ...
```

会让所有 routes 立刻产生大量模型训练，而且相邻 Top-k 高度重叠。

所以 Stage 1 先用 100-feature step 建立“地形”。

只有有竞争力的区域，Stage 2 才支付更细搜索的成本。


In [23]:
def build_candidates(route_orders, sizes, branch):
    subsets = {}
    rows = []
    for route, order in route_orders.items():
        for k in sizes:
            actual_k = min(int(k), len(order))
            name = f"{branch}__{route}__n{actual_k}"
            subsets[name] = list(order[:actual_k])
            rows.append({
                "subset": name,
                "branch": branch,
                "route": route,
                "feature_count": actual_k,
            })
    return subsets, pd.DataFrame(rows)


tree_subsets, tree_catalog = build_candidates(
    tree_routes, COARSE_SIZES, "tree"
)
lr_subsets, lr_catalog = build_candidates(
    lr_routes, COARSE_SIZES, "lr"
)

print("Tree coarse candidates:", len(tree_catalog))
print("LR coarse candidates:", len(lr_catalog))


# Registry keeps the original ranking perspective for every subset.
# Boundary / Fine / Variant / Repair candidates update this registry
# through run_stage1().
subset_route_registry = {
    **dict(
        zip(
            tree_catalog["subset"],
            tree_catalog["route"],
        )
    ),
    **dict(
        zip(
            lr_catalog["subset"],
            lr_catalog["route"],
        )
    ),
}


Tree coarse candidates: 42
LR coarse candidates: 28


## 6. 固定模型基准与公平比较

Tree 与 LR 都在本 Notebook使用固定 baseline。

它们不是最终最优 hyperparameters。

这里固定的原因是：

> **此处旨在 candidate A 和 candidate B 的差异主要来自 feature subset，而不是模型参数也同时发生变化。**

如果每个 subset 都重新调一套参数：

```text
Subset A × 参数组 1...N
Subset B × 参数组 1...N
...
```

Feature Search 会立刻变成：

```text
Feature × Hyperparameter
```

的笛卡尔积。

这样不仅计算成本大，也会增加对 development criterion 的 selection overfitting 风险。

### 但固定 baseline 也不是“无需检查”

如果 baseline 极端 underfit / overfit，subset preference 当然可能被扭曲。

因此冻结前重点看：

- 邻近 k 是否形成 plateau；
- 不同 route 是否产生一致趋势；
- Local Repair 的收益是不是稳定而不是一次性异常；
- F1–F4 是否支持同一片区域。


In [24]:
MISSING_TOKEN = "__MISSING__"
RARE_TOKEN = "__RARE__"

# feature_evidence_report.csv 已经包含 dtype_family
dtype_lookup = (
    feature_evidence
    .set_index("feature")["dtype_family"]
    .to_dict()
)

# 只对本次测速最大 feature pool 中的类别变量学习规则，
# 避免为了测速处理不需要的列。
all_categorical_features = [
    feature
    for feature in candidate_features
    if dtype_lookup.get(feature) in {"categorical", "boolean"}
]

print(
    "Candidate pool:",
    len(candidate_features),
    "features | categorical/boolean:",
    len(all_categorical_features),
)


# ------------------------------------------------------------
# Performance cache: WEEK_NUM per parquet row group
# ------------------------------------------------------------
# SubsetParquetSequence is created many times across candidate subsets/folds.
# WEEK_NUM never changes, so read it once and reuse it.
ROW_GROUP_WEEK_CACHE = [
    train_parquet
    .read_row_group(
        row_group_id,
        columns=[WEEK_COL],
        use_threads=True,
    )
    .column(0)
    .to_numpy(
        zero_copy_only=False
    )
    for row_group_id in range(
        train_parquet.num_row_groups
    )
]

print(
    "Cached WEEK_NUM arrays for",
    len(ROW_GROUP_WEEK_CACHE),
    "row groups.",
)

def learn_fold_category_rules(
    categorical_features,
    week_start,
    week_end,
):
    """
    只在 fold-fit window 学 category levels。
    每个 fold 可以缓存一次，再供不同 subsets 共用。
    """

    from collections import Counter

    rules = {}

    if not categorical_features:
        return rules

    normal_limit = max(
        1,
        LGBM_MAX_CATEGORIES - 2,
    )

    batch_size = 10

    for start in range(
        0,
        len(
            categorical_features
        ),
        batch_size,
    ):

        batch_features = (
            categorical_features[
                start:
                start + batch_size
            ]
        )

        counters = {
            feature: Counter()
            for feature in (
                batch_features
            )
        }

        for row_group_id in range(
            train_parquet.num_row_groups
        ):

            frame = (
                train_parquet
                .read_row_group(
                    row_group_id,
                    columns=[
                        WEEK_COL,
                        *batch_features,
                    ],
                    use_threads=True,
                )
                .to_pandas()
            )

            mask = (
                frame[
                    WEEK_COL
                ]
                .between(
                    week_start,
                    week_end,
                )
            )

            if mask.any():

                selected = (
                    frame.loc[
                        mask,
                        batch_features,
                    ]
                )

                for feature in (
                    batch_features
                ):

                    values = (
                        selected[
                            feature
                        ]
                        .astype("string")
                        .fillna(
                            MISSING_TOKEN
                        )
                    )

                    counts = (
                        values
                        .value_counts()
                    )

                    counters[
                        feature
                    ].update(
                        {
                            str(key): int(value)
                            for key, value
                            in counts.items()
                        }
                    )

            del frame
            gc.collect()

        for feature in (
            batch_features
        ):

            normal_counts = [
                (
                    category,
                    count,
                )
                for category, count
                in counters[
                    feature
                ].items()
                if category
                not in {
                    MISSING_TOKEN,
                    RARE_TOKEN,
                }
            ]

            normal_counts.sort(
                key=lambda item: (
                    -item[1],
                    item[0],
                )
            )

            kept_normal = [
                category
                for category, _
                in normal_counts[
                    :normal_limit
                ]
            ]

            categories = [
                *kept_normal,
                MISSING_TOKEN,
                RARE_TOKEN,
            ]

            rules[
                feature
            ] = {
                "categories": (
                    categories
                ),
                "kept_set": (
                    set(categories)
                ),
            }

    return rules


fold_category_rule_cache = {}


def get_fold_category_rules(
    week_start,
    week_end,
):
    """同一个 fit window 只学习一次 category rules。"""

    key = (
        int(week_start),
        int(week_end),
    )

    if key not in (
        fold_category_rule_cache
    ):

        fold_category_rule_cache[
            key
        ] = (
            learn_fold_category_rules(
                all_categorical_features,
                week_start,
                week_end,
            )
        )

    return (
        fold_category_rule_cache[
            key
        ]
    )




class SubsetParquetSequence(
    lgb.Sequence
):
    """
    当前 subset + 当前 week window 的
    memory-safe LightGBM Sequence。
    """

    batch_size = 4096

    def __init__(
        self,
        parquet_path,
        features,
        categorical_rules,
        week_start,
        week_end,
    ):

        self.parquet_file = (
            pq.ParquetFile(
                parquet_path
            )
        )

        self.features = list(
            features
        )

        self.numeric_features = [
            feature
            for feature
            in self.features
            if dtype_lookup.get(
                feature
            )
            == "numeric"
        ]

        self.categorical_features = [
            feature
            for feature
            in self.features
            if dtype_lookup.get(
                feature
            )
            in {
                "categorical",
                "boolean",
            }
        ]

        self.feature_columns = [
            *self.numeric_features,
            *self.categorical_features,
        ]

        self.categorical_rules = {
            feature: (
                categorical_rules[
                    feature
                ]
            )
            for feature in (
                self.categorical_features
            )
        }

        self.selected_local_indices = []
        selected_sizes = []

        for row_group_id in range(
            self.parquet_file
            .num_row_groups
        ):

            weeks = (
                ROW_GROUP_WEEK_CACHE[
                    row_group_id
                ]
            )

            local_indices = np.flatnonzero(
                (
                    weeks
                    >= week_start
                )
                &
                (
                    weeks
                    <= week_end
                )
            ).astype(
                np.int32
            )

            self.selected_local_indices.append(
                local_indices
            )

            selected_sizes.append(
                len(local_indices)
            )

        self.selected_sizes = np.array(
            selected_sizes,
            dtype=np.int64,
        )

        self.selected_starts = (
            np.concatenate([
                np.array(
                    [0],
                    dtype=np.int64,
                ),
                np.cumsum(
                    self.selected_sizes
                ),
            ])
        )

        self.total_rows = int(
            self.selected_starts[-1]
        )

        self._cache_row_group = None
        self._cache_matrix = None


    def __len__(self):
        return self.total_rows


    def _convert_row_group(
        self,
        row_group_id,
    ):

        local_indices = (
            self.selected_local_indices[
                row_group_id
            ]
        )

        if len(
            local_indices
        ) == 0:

            return np.empty(
                (
                    0,
                    len(
                        self.feature_columns
                    ),
                ),
                dtype=np.float64,
            )

        table = (
            self.parquet_file
            .read_row_group(
                row_group_id,
                columns=(
                    self.feature_columns
                ),
                use_threads=True,
            )
            .take(
                pa.array(
                    local_indices
                )
            )
        )

        frame = (
            table.to_pandas()
        )

        if self.numeric_features:

            numerical = (
                frame[
                    self.numeric_features
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce",
                )
                .to_numpy(
                    dtype=np.float64,
                    na_value=np.nan,
                )
            )

        else:

            numerical = np.empty(
                (
                    len(frame),
                    0,
                ),
                dtype=np.float64,
            )

        categorical_arrays = []

        for feature in (
            self.categorical_features
        ):

            values = (
                frame[
                    feature
                ]
                .astype("string")
                .fillna(
                    MISSING_TOKEN
                )
            )

            rule = (
                self.categorical_rules[
                    feature
                ]
            )

            values = values.where(
                values.isin(
                    rule[
                        "kept_set"
                    ]
                ),
                RARE_TOKEN,
            )

            codes = (
                pd.Categorical(
                    values,
                    categories=(
                        rule[
                            "categories"
                        ]
                    ),
                )
                .codes
                .astype(
                    np.float64
                )
            )

            categorical_arrays.append(
                codes
            )

        if categorical_arrays:

            categorical_matrix = (
                np.column_stack(
                    categorical_arrays
                )
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            matrix = np.hstack([
                numerical,
                categorical_matrix,
            ])

        else:

            matrix = numerical

        del (
            table,
            frame,
        )

        gc.collect()

        return np.asarray(
            matrix,
            dtype=np.float64,
            order="C",
        )


    def _get_row_group_matrix(
        self,
        row_group_id,
    ):

        if (
            self._cache_row_group
            != row_group_id
        ):

            self._cache_matrix = (
                self._convert_row_group(
                    row_group_id
                )
            )

            self._cache_row_group = (
                row_group_id
            )

        return (
            self._cache_matrix
        )


    def _locate_row_group(
        self,
        selected_row_index,
    ):

        row_group_id = int(
            np.searchsorted(
                self.selected_starts[
                    1:
                ],
                selected_row_index,
                side="right",
            )
        )

        local_selected_index = int(
            selected_row_index
            - self.selected_starts[
                row_group_id
            ]
        )

        return (
            row_group_id,
            local_selected_index,
        )


    def __getitem__(
        self,
        idx,
    ):

        if isinstance(
            idx,
            (int, np.integer),
        ):

            if idx < 0:
                idx += len(self)

            (
                row_group_id,
                local_selected,
            ) = (
                self._locate_row_group(
                    int(idx)
                )
            )

            matrix = (
                self._get_row_group_matrix(
                    row_group_id
                )
            )

            return matrix[
                local_selected
            ]

        if isinstance(
            idx,
            slice,
        ):

            start = (
                0
                if idx.start is None
                else int(idx.start)
            )

            stop = (
                len(self)
                if idx.stop is None
                else int(idx.stop)
            )

            parts = []
            current = start

            while current < stop:

                (
                    row_group_id,
                    local_selected,
                ) = (
                    self._locate_row_group(
                        current
                    )
                )

                matrix = (
                    self._get_row_group_matrix(
                        row_group_id
                    )
                )

                available = (
                    len(matrix)
                    - local_selected
                )

                take = min(
                    available,
                    stop - current,
                )

                parts.append(
                    matrix[
                        local_selected:
                        local_selected
                        + take
                    ]
                )

                current += take

            if not parts:

                return np.empty(
                    (
                        0,
                        len(
                            self.feature_columns
                        ),
                    ),
                    dtype=np.float64,
                )

            if len(parts) == 1:
                return parts[0]

            return np.vstack(
                parts
            )

        if isinstance(
            idx,
            (
                list,
                np.ndarray,
            ),
        ):

            return np.stack([
                self[int(i)]
                for i in idx
            ])

        raise TypeError(
            "Unsupported index type："
            f"{type(idx)}"
        )




def fit_streaming_booster(
    features,
    train_start,
    train_end,
):
    """固定参数 LightGBM，memory-safe fit。"""

    import lightgbm as lgb

    category_rules = (
        get_fold_category_rules(
            train_start,
            train_end,
        )
    )

    sequence = (
        SubsetParquetSequence(
            TRAIN_PATH,
            features,
            category_rules,
            week_start=(
                train_start
            ),
            week_end=(
                train_end
            ),
        )
    )

    fit_mask = (
        train_meta[
            WEEK_COL
        ]
        .between(
            train_start,
            train_end,
        )
    )

    y = (
        train_meta.loc[
            fit_mask,
            TARGET_COL,
        ]
        .to_numpy(
            dtype=np.int8
        )
    )

    assert (
        len(sequence)
        == len(y)
    )

    feature_names = (
        sequence.feature_columns
    )

    categorical_indices = list(
        range(
            len(
                sequence.numeric_features
            ),
            len(
                feature_names
            ),
        )
    )

    dataset = lgb.Dataset(
        sequence,
        label=y,
        feature_name=(
            feature_names
        ),
        categorical_feature=(
            categorical_indices
        ),
        free_raw_data=True,
        params={
            "bin_construct_sample_cnt": (
                LGBM_BIN_SAMPLE_COUNT
            ),
            "max_bin": (
                LGBM_MAX_BIN
            ),
            "enable_bundle": (
                LGBM_ENABLE_BUNDLE
            ),
            "data_random_seed": (
                RANDOM_SEED
            ),
        },
    )

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": (
            float(
                nb8_config.get(
                    "lgbm_learning_rate",
                    0.05,
                )
            )
        ),
        "num_leaves": int(
            nb8_config.get(
                "lgbm_num_leaves",
                31,
            )
        ),
        "feature_fraction": float(
            nb8_config.get(
                "lgbm_colsample_bytree",
                0.8,
            )
        ),
        "bagging_fraction": float(
            nb8_config.get(
                "lgbm_subsample",
                0.8,
            )
        ),
        "bagging_freq": int(
            nb8_config.get(
                "lgbm_subsample_freq",
                1,
            )
        ),
        "lambda_l2": float(
            nb8_config.get(
                "lgbm_reg_lambda",
                1.0,
            )
        ),
        "seed": RANDOM_SEED,
        "verbosity": -1,
        "num_threads": -1,
        "device_type": DEVICE_TYPE,
    }

    booster = lgb.train(
        params=params,
        train_set=dataset,
        num_boost_round=(
            LGBM_NUM_BOOST_ROUND
        ),
    )

    del (
        dataset,
        sequence,
        y,
    )

    gc.collect()

    return booster


def predict_streaming_window(
    booster,
    features,
    fit_start,
    fit_end,
    assess_start,
    assess_end,
):
    """用 fit-window category rules 对 assessment window 分批预测。"""

    category_rules = (
        get_fold_category_rules(
            fit_start,
            fit_end,
        )
    )

    sequence = (
        SubsetParquetSequence(
            TRAIN_PATH,
            features,
            category_rules,
            week_start=(
                assess_start
            ),
            week_end=(
                assess_end
            ),
        )
    )

    predictions = []

    batch_size = 8192

    for start in range(
        0,
        len(sequence),
        batch_size,
    ):

        stop = min(
            start + batch_size,
            len(sequence),
        )

        matrix = sequence[
            start:stop
        ]

        predictions.append(
            booster.predict(
                matrix
            )
        )

        del matrix

    prediction = np.concatenate(
        predictions
    )

    del (
        sequence,
        predictions,
    )

    gc.collect()

    return prediction


Candidate pool: 1209 features | categorical/boolean: 59
Cached WEEK_NUM arrays for 153 row groups.


In [25]:
# ============================================================
# Logistic Regression streaming evaluator
# 复用 Notebook 7 已冻结 preprocessing 规则。
# ============================================================
LR_MISSING_TOKEN = "__HC_MISSING__"
LR_RARE_TOKEN = "__HC_RARE__"

nb7_dtype = nb7_catalog.set_index("feature")["dtype_family"].to_dict()
numeric_stats_indexed = numeric_stats.set_index("feature")


def make_lr_state(features):
    numeric = [f for f in features if nb7_dtype.get(f) == "numeric"]
    categorical = [
        f for f in features
        if nb7_dtype.get(f) in {"categorical", "boolean"}
    ]

    stats = numeric_stats_indexed.reindex(numeric)
    if numeric and stats[["median", "scale_after_imputation"]].isna().any().any():
        missing = stats.index[stats["median"].isna()].tolist()
        raise ValueError(f"LR numerical preprocessing rules missing: {missing[:10]}")

    medians = stats["median"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)

    means = (
        stats["mean_after_imputation"].to_numpy(dtype=np.float32)
        if numeric else np.array([], dtype=np.float32)
    )
    scales = stats["scale_after_imputation"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)

    missing_features = (
        stats.index[stats["has_missing"].fillna(False).astype(bool)].tolist()
        if numeric else []
    )
    missing_indices = [numeric.index(f) for f in missing_features]

    category_lists = []
    kept_sets = {}
    for f in categorical:
        cats = [str(x) for x in lr_category_lists[f]]
        category_lists.append(cats)
        kept_sets[f] = set(cats)

    if categorical:
        kwargs = dict(
            categories=category_lists,
            handle_unknown="ignore",
            dtype=np.float32,
        )
        try:
            encoder = OneHotEncoder(sparse_output=True, **kwargs)
        except TypeError:
            encoder = OneHotEncoder(sparse=True, **kwargs)

        dummy = pd.DataFrame({f: [category_lists[i][0]] for i, f in enumerate(categorical)})
        encoder.fit(dummy)
    else:
        encoder = None

    return {
        "features": list(features),
        "numeric": numeric,
        "categorical": categorical,
        "medians": medians,
        "means": means,
        "scales": scales,
        "missing_indices": missing_indices,
        "kept_sets": kept_sets,
        "encoder": encoder,
    }


def transform_lr_frame(frame, state):
    n = len(frame)

    numeric = state["numeric"]
    if numeric:
        X_num = frame[numeric].to_numpy(dtype=np.float32, na_value=np.nan)
        missing_mask = ~np.isfinite(X_num)
        if missing_mask.any():
            rows, cols = np.where(missing_mask)
            X_num[rows, cols] = state["medians"][cols]
        X_num -= state["means"]
        X_num /= state["scales"]
        num_sparse = sparse.csr_matrix(X_num, dtype=np.float32)

        if state["missing_indices"]:
            X_miss = missing_mask[:, state["missing_indices"]].astype(np.float32)
            miss_sparse = sparse.csr_matrix(X_miss, dtype=np.float32)
        else:
            miss_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)
    else:
        num_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)
        miss_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)

    categorical = state["categorical"]
    if categorical:
        mapped = {}
        for f in categorical:
            s = frame[f].astype("string").fillna(LR_MISSING_TOKEN)
            kept = state["kept_sets"][f]
            s = s.where(s.isin(kept), LR_RARE_TOKEN)
            mapped[f] = s.astype(object)
        mapped = pd.DataFrame(mapped, index=frame.index)
        cat_sparse = state["encoder"].transform(mapped)
    else:
        cat_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)

    return sparse.hstack(
        [num_sparse, miss_sparse, cat_sparse],
        format="csr",
        dtype=np.float32,
    )


def fit_streaming_lr(features, train_start, train_end):
    state = make_lr_state(features)
    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=LR_ALPHA,
        learning_rate="constant",
        eta0=LR_ETA0,
        shuffle=False,
        average=True,
        random_state=RANDOM_SEED,
    )

    first = True
    parquet = pq.ParquetFile(TRAIN_PATH)

    for epoch in range(LR_EPOCHS):
        row_groups = np.arange(parquet.num_row_groups)
        rng = np.random.default_rng(RANDOM_SEED + epoch)
        rng.shuffle(row_groups)

        for rg in row_groups:
            frame = parquet.read_row_group(
                int(rg),
                columns=[WEEK_COL, TARGET_COL, *features],
                use_threads=True,
            ).to_pandas()

            frame = frame.loc[
                frame[WEEK_COL].between(train_start, train_end)
            ]
            if frame.empty:
                continue

            # deterministic whole-row shuffle
            frame = frame.sample(
                frac=1.0,
                random_state=RANDOM_SEED + epoch * 1000 + int(rg),
            ).reset_index(drop=True)

            y = frame[TARGET_COL].to_numpy(dtype=np.int8)
            X = transform_lr_frame(frame, state)

            if first:
                model.partial_fit(X, y, classes=np.array([0, 1], dtype=np.int8))
                first = False
            else:
                model.partial_fit(X, y)

            del frame, X, y
            gc.collect()

    return model, state


def predict_streaming_lr(model, state, features, assess_start, assess_end):
    parquet = pq.ParquetFile(TRAIN_PATH)
    predictions = []

    for rg in range(parquet.num_row_groups):
        frame = parquet.read_row_group(
            rg,
            columns=[WEEK_COL, *features],
            use_threads=True,
        ).to_pandas()
        frame = frame.loc[
            frame[WEEK_COL].between(assess_start, assess_end)
        ].reset_index(drop=True)
        if frame.empty:
            continue

        X = transform_lr_frame(frame, state)
        pred = model.predict_proba(X)[:, 1]
        predictions.append(pred.astype(np.float32))
        del frame, X, pred
        gc.collect()

    return np.concatenate(predictions)


In [26]:
def calculate_weekly_gini(
    prediction_frame,
):
    """逐周计算 AUC → Gini。"""

    rows = []

    for week, group in (
        prediction_frame
        .groupby(
            WEEK_COL,
            sort=True,
        )
    ):

        # 某一周如果 target 只有一个类别，
        # AUC 无法定义。
        if (
            group[
                TARGET_COL
            ]
            .nunique()
            < 2
        ):

            gini = np.nan

        else:

            auc = roc_auc_score(
                group[
                    TARGET_COL
                ],
                group[
                    "prediction"
                ],
            )

            gini = (
                2.0 * auc
                - 1.0
            )

        rows.append({
            WEEK_COL: int(
                week
            ),
            "gini": (
                gini
            ),
            "sample_count": (
                len(
                    group
                )
            ),
        })

    return pd.DataFrame(
        rows
    )


def temporal_metrics(
    weekly_gini,
):
    """
    把 weekly Gini 压缩成：
    Mean Gini / slope / residual SD。
    """

    table = (
        weekly_gini
        .dropna(
            subset=[
                "gini"
            ]
        )
        .sort_values(
            WEEK_COL
        )
    )

    if len(table) < 3:

        return {
            "mean_weekly_gini": (
                np.nan
            ),
            "weekly_gini_slope": (
                np.nan
            ),
            "negative_slope": (
                np.nan
            ),
            "weekly_residual_sd": (
                np.nan
            ),
            "stability_metric": (
                np.nan
            ),
        }

    x = (
        table[
            WEEK_COL
        ]
        .to_numpy(
            dtype=float
        )
    )

    y = (
        table[
            "gini"
        ]
        .to_numpy(
            dtype=float
        )
    )

    slope, intercept = (
        np.polyfit(
            x,
            y,
            deg=1,
        )
    )

    fitted = (
        slope * x
        + intercept
    )

    residual_sd = float(
        np.std(
            y - fitted,
            ddof=0,
        )
    )

    mean_gini = float(
        np.mean(
            y
        )
    )

    negative_slope = max(
        0.0,
        -float(
            slope
        ),
    )

    stability_metric = (
        mean_gini
        + 88.0
        * min(
            0.0,
            float(
                slope
            ),
        )
        - 0.5
        * residual_sd
    )

    return {
        "mean_weekly_gini": (
            mean_gini
        ),
        "weekly_gini_slope": (
            float(
                slope
            )
        ),
        "negative_slope": (
            negative_slope
        ),
        "weekly_residual_sd": (
            residual_sd
        ),
        "stability_metric": (
            stability_metric
        ),
    }


PARETO_OBJECTIVES = {
    "mean_weekly_gini": (
        "max"
    ),
    "negative_slope": (
        "min"
    ),
    "weekly_residual_sd": (
        "min"
    ),
    "feature_count": (
        "min"
    ),
}


def dominates(
    row_a,
    row_b,
):
    """
    A dominates B：
    所有目标都不差，
    而且至少一个目标更好。
    """

    no_worse = True
    strictly_better = False

    for column, direction in (
        PARETO_OBJECTIVES.items()
    ):

        a = row_a[
            column
        ]

        b = row_b[
            column
        ]

        if (
            pd.isna(a)
            or
            pd.isna(b)
        ):
            return False

        if direction == "max":

            if a < b:
                no_worse = False
                break

            if a > b:
                strictly_better = True

        else:

            if a > b:
                no_worse = False
                break

            if a < b:
                strictly_better = True

    return (
        no_worse
        and
        strictly_better
    )


def pareto_rank(
    result_table,
):
    """给每个 subset 标记 Pareto rank。"""

    table = (
        result_table
        .reset_index(
            drop=True
        )
        .copy()
    )

    remaining = set(
        table.index
    )

    ranks = pd.Series(
        np.nan,
        index=table.index,
    )

    current_rank = 1

    while remaining:

        front = []

        for i in list(
            remaining
        ):

            is_dominated = any(
                dominates(
                    table.loc[j],
                    table.loc[i],
                )
                for j in remaining
                if j != i
            )

            if not is_dominated:

                front.append(
                    i
                )

        for i in front:

            ranks.loc[
                i
            ] = current_rank

            remaining.remove(
                i
            )

        current_rank += 1

    table[
        "pareto_rank"
    ] = ranks.astype(
        int
    )

    table[
        "pareto_front"
    ] = (
        table[
            "pareto_rank"
        ]
        .eq(1)
    )

    return table


def crowding_distance(
    front,
):
    """
    如果一个 Pareto front 太大，
    用 crowding distance 保留不同 trade-off 区域。
    """

    front = (
        front
        .copy()
    )

    distance = pd.Series(
        0.0,
        index=front.index,
    )

    if len(front) <= 2:

        distance[:] = np.inf

        return distance

    for column in (
        PARETO_OBJECTIVES
    ):

        ordered = (
            front[
                column
            ]
            .sort_values()
            .index
            .tolist()
        )

        distance.loc[
            ordered[0]
        ] = np.inf

        distance.loc[
            ordered[-1]
        ] = np.inf

        low = float(
            front.loc[
                ordered[0],
                column,
            ]
        )

        high = float(
            front.loc[
                ordered[-1],
                column,
            ]
        )

        span = (
            high - low
        )

        if span <= 0:
            continue

        for position in range(
            1,
            len(ordered) - 1,
        ):

            current = (
                ordered[
                    position
                ]
            )

            if np.isinf(
                distance.loc[
                    current
                ]
            ):
                continue

            previous_value = (
                front.loc[
                    ordered[
                        position - 1
                    ],
                    column,
                ]
            )

            next_value = (
                front.loc[
                    ordered[
                        position + 1
                    ],
                    column,
                ]
            )

            distance.loc[
                current
            ] += (
                abs(
                    next_value
                    - previous_value
                )
                / span
            )

    return distance


def select_survivors(
    result_table,
    budget,
):
    """
    先按 Pareto rank 收。
    最后一个装不下的 front，
    再用 crowding distance 截断。
    """

    ranked = pareto_rank(
        result_table
    )

    selected = []

    for rank in sorted(
        ranked[
            "pareto_rank"
        ].unique()
    ):

        front = ranked.loc[
            ranked[
                "pareto_rank"
            ].eq(
                rank
            )
        ]

        remaining_slots = (
            int(
                budget
            )
            - len(
                selected
            )
        )

        if remaining_slots <= 0:
            break

        if (
            len(front)
            <= remaining_slots
        ):

            selected.extend(
                front.index.tolist()
            )

        else:

            distance = (
                crowding_distance(
                    front
                )
            )

            chosen = (
                distance
                .sort_values(
                    ascending=False
                )
                .head(
                    remaining_slots
                )
                .index
                .tolist()
            )

            selected.extend(
                chosen
            )

            break

    survivors = (
        ranked.loc[
            selected
        ]
        .copy()
    )

    return (
        ranked,
        survivors,
    )


## 7. Stage-1 Evaluation — 候选子集比较

所有 coarse candidates 使用同一时间边界：

```text
Train  0–55
Assess 56–63
```

每个 subset 保存：

```text
AUC / Gini
Mean Weekly Gini
Negative Slope
Residual SD
Feature Count
```

本 Notebook不把这些目标强行加成一个 FinalScore。

而是使用 Pareto dominance：

```text
Mean Weekly Gini   ↑
Negative Slope     ↓
Residual SD        ↓
Feature Count      ↓
```

### 为什么要 Pareto？

因为：

```text
更高预测力
更稳定
更少 features
```

可能互相冲突。

一个 subset 多 100 个 features，只换来极小 Gini 提升，它不一定应该自动“赢”。


In [27]:
STAGE1_TRAIN_START = 0
STAGE1_TRAIN_END = 55
STAGE1_ASSESS_START = 56
STAGE1_ASSESS_END = 63


def make_prediction_frame(pred, assess_start, assess_end):
    mask = train_meta[WEEK_COL].between(assess_start, assess_end)
    out = train_meta.loc[
        mask, [ID_COL, WEEK_COL, TARGET_COL]
    ].reset_index(drop=True)
    if len(out) != len(pred):
        raise ValueError(
            f"Prediction alignment mismatch: meta={len(out)}, pred={len(pred)}"
        )
    out["prediction"] = pred
    return out


def _candidate_cache_key(
    branch,
    subset_name,
    features,
    train_start,
    train_end,
    assess_start,
    assess_end,
):
    """候选 subset + 时间窗 + fixed baseline 的稳定 fingerprint。"""

    model_config = {
        "branch": branch,
        "subset_name": subset_name,
        "features": list(features),
        "train_start": int(train_start),
        "train_end": int(train_end),
        "assess_start": int(assess_start),
        "assess_end": int(assess_end),
        "random_seed": RANDOM_SEED,
        "lgbm_num_boost_round": LGBM_NUM_BOOST_ROUND,
        "lgbm_bin_sample_count": LGBM_BIN_SAMPLE_COUNT,
        "lgbm_max_bin": LGBM_MAX_BIN,
        "device_type": DEVICE_TYPE,
        "lr_epochs": LR_EPOCHS,
        "lr_alpha": LR_ALPHA,
        "lr_eta0": LR_ETA0,
    }

    payload = json.dumps(
        model_config,
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()[:20]


def evaluate_candidate(
    branch,
    subset_name,
    features,
    train_start,
    train_end,
    assess_start,
    assess_end,
):
    """Wrapper evaluation with resumable checkpoint."""

    cache_key = _candidate_cache_key(
        branch,
        subset_name,
        features,
        train_start,
        train_end,
        assess_start,
        assess_end,
    )

    summary_path = (
        CHECKPOINT_ROOT
        / f"{branch}__{cache_key}__summary.json"
    )
    weekly_path = (
        CHECKPOINT_ROOT
        / f"{branch}__{cache_key}__weekly.csv"
    )

    if summary_path.exists() and weekly_path.exists():
        with open(summary_path, "r", encoding="utf-8") as file:
            summary = json.load(file)
        weekly = pd.read_csv(weekly_path)
        print("  checkpoint hit:", subset_name)
        return summary, weekly

    started = time.perf_counter()

    if branch == "tree":
        model = fit_streaming_booster(
            features,
            train_start,
            train_end,
        )
        pred = predict_streaming_window(
            model,
            features,
            fit_start=train_start,
            fit_end=train_end,
            assess_start=assess_start,
            assess_end=assess_end,
        )
        del model

    elif branch == "lr":
        model, state = fit_streaming_lr(
            features,
            train_start,
            train_end,
        )
        pred = predict_streaming_lr(
            model, state, features, assess_start, assess_end
        )
        del model, state

    else:
        raise ValueError(branch)

    frame = make_prediction_frame(
        pred, assess_start, assess_end
    )
    auc = roc_auc_score(
        frame[TARGET_COL], frame["prediction"]
    )
    weekly = calculate_weekly_gini(frame)
    temporal = temporal_metrics(weekly)

    summary = {
        "subset": subset_name,
        "branch": branch,
        "feature_count": int(len(features)),
        "auc": float(auc),
        "gini": float(2 * auc - 1),
        "fit_seconds": float(time.perf_counter() - started),
        **{
            key: (float(value) if pd.notna(value) else None)
            for key, value in temporal.items()
        },
    }

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False, indent=2)
    weekly.to_csv(weekly_path, index=False)

    del pred, frame
    gc.collect()
    return summary, weekly

def run_stage1(branch, catalog, subset_dict):
    rows, weekly_parts = [], []
    for row in catalog.itertuples(index=False):
        name = row.subset
        features = subset_dict[name]
        print(f"{branch} Stage-1: {name} | {len(features)} features")
        summary, weekly = evaluate_candidate(
            branch, name, features,
            STAGE1_TRAIN_START, STAGE1_TRAIN_END,
            STAGE1_ASSESS_START, STAGE1_ASSESS_END,
        )
        summary["route"] = row.route
        subset_route_registry[
            name
        ] = row.route
        rows.append(summary)
        weekly["subset"] = name
        weekly_parts.append(weekly)

    summary = pd.DataFrame(rows)
    ranked = pareto_rank(summary)
    weekly = pd.concat(weekly_parts, ignore_index=True) if weekly_parts else pd.DataFrame()
    return ranked, weekly


tree_coarse_ranked = pd.DataFrame()
lr_coarse_ranked = pd.DataFrame()
tree_coarse_weekly = pd.DataFrame()
lr_coarse_weekly = pd.DataFrame()

if RUN_TREE_COARSE:
    tree_coarse_ranked, tree_coarse_weekly = run_stage1(
        "tree", tree_catalog, tree_subsets
    )
    display(tree_coarse_ranked.sort_values(["pareto_rank", "feature_count"]))

if RUN_LR_COARSE:
    lr_coarse_ranked, lr_coarse_weekly = run_stage1(
        "lr", lr_catalog, lr_subsets
    )
    display(lr_coarse_ranked.sort_values(["pareto_rank", "feature_count"]))


tree Stage-1: tree__relevance__n200 | 200 features
  checkpoint hit: tree__relevance__n200
tree Stage-1: tree__relevance__n300 | 300 features
  checkpoint hit: tree__relevance__n300
tree Stage-1: tree__relevance__n400 | 400 features
  checkpoint hit: tree__relevance__n400
tree Stage-1: tree__relevance__n500 | 500 features
  checkpoint hit: tree__relevance__n500
tree Stage-1: tree__relevance__n600 | 600 features
  checkpoint hit: tree__relevance__n600
tree Stage-1: tree__relevance__n700 | 700 features
  checkpoint hit: tree__relevance__n700
tree Stage-1: tree__relevance__n800 | 800 features
  checkpoint hit: tree__relevance__n800
tree Stage-1: tree__model_gain__n200 | 200 features
  checkpoint hit: tree__model_gain__n200
tree Stage-1: tree__model_gain__n300 | 300 features
  checkpoint hit: tree__model_gain__n300
tree Stage-1: tree__model_gain__n400 | 400 features
  checkpoint hit: tree__model_gain__n400
tree Stage-1: tree__model_gain__n500 | 500 features
  checkpoint hit: tree__model_ga

,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
7,tree__model_gain__n200,tree,200,0.842505,0.685010,110.339146,0.685145,-0.000922,0.000922,0.018868,0.594555,model_gain,1,True
14,tree__temporal_relevance__n200,tree,200,0.815655,0.631310,90.811396,0.632039,0.001236,0.000000,0.021220,0.621429,temporal_relevance,1,True
28,tree__redundancy_aware__n200,tree,200,0.823818,0.647636,90.277235,0.647963,-0.000957,0.000957,0.016737,0.555407,redundancy_aware,1,True
35,tree__consensus__n200,tree,200,0.834295,0.668589,91.983092,0.668750,-0.000486,0.000486,0.017936,0.617054,consensus,1,True
8,tree__model_gain__n300,tree,300,0.843130,0.686260,129.147651,0.686393,-0.000910,0.000910,0.018524,0.597026,model_gain,1,True
15,tree__temporal_relevance__n300,tree,300,0.837364,0.674728,105.918940,0.674763,-0.000186,0.000186,0.019455,0.648708,temporal_relevance,1,True
22,tree__interaction_relevance__n300,tree,300,0.824449,0.648898,103.821927,0.649334,-0.000883,0.000883,0.017335,0.562936,interaction_relevance,1,True
29,tree__redundancy_aware__n300,tree,300,0.837380,0.674760,112.040646,0.674919,-0.001077,0.001077,0.016102,0.572129,redundancy_aware,1,True
36,tree__consensus__n300,tree,300,0.835452,0.670903,110.922510,0.670939,-0.000992,0.000992,0.017322,0.574945,consensus,1,True
9,tree__model_gain__n400,tree,400,0.843571,0.687141,142.910468,0.687307,-0.000726,0.000726,0.019105,0.613880,model_gain,1,True


lr Stage-1: lr__relevance__n200 | 200 features
  checkpoint hit: lr__relevance__n200
lr Stage-1: lr__relevance__n300 | 300 features
  checkpoint hit: lr__relevance__n300
lr Stage-1: lr__relevance__n400 | 400 features
  checkpoint hit: lr__relevance__n400
lr Stage-1: lr__relevance__n500 | 500 features
  checkpoint hit: lr__relevance__n500
lr Stage-1: lr__relevance__n600 | 600 features
  checkpoint hit: lr__relevance__n600
lr Stage-1: lr__relevance__n700 | 700 features
  checkpoint hit: lr__relevance__n700
lr Stage-1: lr__relevance__n800 | 800 features
  checkpoint hit: lr__relevance__n800
lr Stage-1: lr__temporal_relevance__n200 | 200 features
  checkpoint hit: lr__temporal_relevance__n200
lr Stage-1: lr__temporal_relevance__n300 | 300 features
  checkpoint hit: lr__temporal_relevance__n300
lr Stage-1: lr__temporal_relevance__n400 | 400 features
  checkpoint hit: lr__temporal_relevance__n400
lr Stage-1: lr__temporal_relevance__n500 | 500 features
  checkpoint hit: lr__temporal_relevance

,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
0,lr__relevance__n200,lr,200,0.736753,0.473506,148.562161,0.473212,-0.001315,0.001315,0.026801,0.344069,relevance,1,True
7,lr__temporal_relevance__n200,lr,200,0.718434,0.436868,151.564201,0.436400,-0.002589,0.002589,0.022534,0.197320,temporal_relevance,1,True
14,lr__redundancy_aware__n200,lr,200,0.718389,0.436778,160.563415,0.435911,-0.007642,0.007642,0.016162,-0.244706,redundancy_aware,1,True
21,lr__lr_consensus__n200,lr,200,0.716776,0.433552,155.946993,0.432416,-0.004711,0.004711,0.017033,0.009298,lr_consensus,1,True
8,lr__temporal_relevance__n300,lr,300,0.727821,0.455642,189.506828,0.453747,-0.007722,0.007722,0.020253,-0.235887,temporal_relevance,1,True
15,lr__redundancy_aware__n300,lr,300,0.729131,0.458263,212.400993,0.456384,-0.011293,0.011293,0.013965,-0.544359,redundancy_aware,1,True
23,lr__lr_consensus__n400,lr,400,0.704743,0.409486,263.484940,0.407251,-0.009664,0.009664,0.015402,-0.450851,lr_consensus,1,True
3,lr__relevance__n500,lr,500,0.718404,0.436808,313.628939,0.434935,-0.013607,0.013607,0.013953,-0.769469,relevance,1,True
17,lr__redundancy_aware__n500,lr,500,0.713773,0.427546,354.172305,0.425557,-0.011442,0.011442,0.013124,-0.587910,redundancy_aware,1,True
24,lr__lr_consensus__n500,lr,500,0.692154,0.384308,319.002865,0.381948,-0.010324,0.010324,0.013503,-0.533350,lr_consensus,1,True


## 8. Stage 2A — Boundary Check

Coarse Search 的默认范围：

```text
200 ... 800
```

只是初始计算预算。

它不是理论上证明：

```text
最优 feature count 一定在 200–800
```

如果出现：

```text
700 < 800
```

而 800 仍然在 Pareto front，不能立刻说：

> “800 最优。”

因为 800 只是初始搜索设置的停止位置。

这与只采用：

```text
300 / 500 / 800
```

的问题本质相同，只是网格更密而已。

所以 Boundary Check 问：

> **当前竞争力最强的区域是否仍然贴着搜索边界？**

如果是，才支付额外成本向外扩展。

例如：

```text
800
→ 900
```

如果 900 继续改善，最多再进行有限轮次扩展。

如果已经 plateau / deteriorate，就停止。

### 成本问题意识

增加 boundary models 针对的是：

> **人工边界造成的 false optimum。**

限制 `MAX_BOUNDARY_ROUNDS` 针对的是：

> **无限扩张搜索范围造成新的 experiment explosion。**

> 本版中 Boundary Check 不再只是“给出建议”。当 `RUN_BOUNDARY_EXPANSION=True` 时，proposal 会真正进入与 coarse/fine 相同的 wrapper evaluation，并且同样使用 checkpoint。


In [28]:
# ============================================================
# Stage 2A — Boundary diagnostics / candidate generation
# ============================================================

def boundary_candidates(
    coarse_ranked,
    route_orders,
    existing_subsets,
    branch,
):
    """
    只在 Pareto front 仍贴着 coarse-search 边界时，
    提出向外扩展的 candidate。

    这里先生成 proposal，不默认运行模型。
    """
    if coarse_ranked.empty:
        return {}, pd.DataFrame()

    front = (
        coarse_ranked
        .loc[
            coarse_ranked[
                "pareto_front"
            ]
        ]
        .copy()
    )

    if front.empty:
        return {}, pd.DataFrame()

    coarse_min = min(
        COARSE_SIZES
    )

    coarse_max = max(
        COARSE_SIZES
    )

    proposals = {}
    rows = []

    for row in front.itertuples(
        index=False
    ):
        route = row.route
        k = int(
            row.feature_count
        )

        if route not in route_orders:
            continue

        proposed_sizes = []

        if k <= coarse_min:
            proposed_sizes.append(
                k
                - BOUNDARY_EXPANSION_STEP
            )

        if k >= coarse_max:
            proposed_sizes.append(
                k
                + BOUNDARY_EXPANSION_STEP
            )

        for proposed_k in proposed_sizes:

            if not (
                MIN_SEARCH_SIZE
                <= proposed_k
                <= MAX_SEARCH_SIZE
            ):
                continue

            order = route_orders[
                route
            ]

            actual_k = min(
                proposed_k,
                len(order),
            )

            name = (
                f"{branch}__{route}"
                f"__boundary_n{actual_k}"
            )

            if (
                name in existing_subsets
                or name in proposals
            ):
                continue

            proposals[
                name
            ] = list(
                order[
                    :actual_k
                ]
            )

            rows.append({
                "subset": name,
                "branch": branch,
                "route": route,
                "feature_count": (
                    actual_k
                ),
                "reason": (
                    "pareto_front_at_"
                    "search_boundary"
                ),
            })

    return (
        proposals,
        pd.DataFrame(
            rows
        ),
    )


tree_boundary_proposals = (
    {},
    pd.DataFrame(),
)

lr_boundary_proposals = (
    {},
    pd.DataFrame(),
)


if not tree_coarse_ranked.empty:
    tree_boundary_proposals = (
        boundary_candidates(
            tree_coarse_ranked,
            tree_routes,
            tree_subsets,
            "tree",
        )
    )

if not lr_coarse_ranked.empty:
    lr_boundary_proposals = (
        boundary_candidates(
            lr_coarse_ranked,
            lr_routes,
            lr_subsets,
            "lr",
        )
    )


print(
    "Tree boundary proposals:"
)

display(
    tree_boundary_proposals[1]
)

print(
    "LR boundary proposals:"
)

display(
    lr_boundary_proposals[1]
)


Tree boundary proposals:


,subset,branch,route,feature_count,reason
0,tree__relevance__boundary_n900,tree,relevance,900,pareto_front_at_search_boundary
1,tree__model_gain__boundary_n100,tree,model_gain,100,pareto_front_at_search_boundary
2,tree__temporal_relevance__boundary_n100,tree,temporal_relevance,100,pareto_front_at_search_boundary
3,tree__redundancy_aware__boundary_n100,tree,redundancy_aware,100,pareto_front_at_search_boundary
4,tree__redundancy_aware__boundary_n900,tree,redundancy_aware,900,pareto_front_at_search_boundary
5,tree__consensus__boundary_n100,tree,consensus,100,pareto_front_at_search_boundary
6,tree__consensus__boundary_n900,tree,consensus,900,pareto_front_at_search_boundary


LR boundary proposals:


,subset,branch,route,feature_count,reason
0,lr__relevance__boundary_n100,lr,relevance,100,pareto_front_at_search_boundary
1,lr__relevance__boundary_n900,lr,relevance,900,pareto_front_at_search_boundary
2,lr__temporal_relevance__boundary_n100,lr,temporal_relevance,100,pareto_front_at_search_boundary
3,lr__temporal_relevance__boundary_n900,lr,temporal_relevance,900,pareto_front_at_search_boundary
4,lr__redundancy_aware__boundary_n100,lr,redundancy_aware,100,pareto_front_at_search_boundary
5,lr__redundancy_aware__boundary_n900,lr,redundancy_aware,900,pareto_front_at_search_boundary
6,lr__lr_consensus__boundary_n100,lr,lr_consensus,100,pareto_front_at_search_boundary
7,lr__lr_consensus__boundary_n900,lr,lr_consensus,900,pareto_front_at_search_boundary


## 9. Stage 2B — Fine Search

Coarse 已经建立：

```text
200 / 300 / ... / 800
```

Fine Search 不重新全局扫描。

只对 Pareto anchors 补：

```text
k - 50
k + 50
```

例如：

```text
coarse Pareto:
k = 500

fine:
450 / 550
```

它解决的是：

> **100-feature step 仍然可能太粗。**

但这里仍然只改变：

```text
Depth
```

不改变 subset membership rule。

所以 Fine Search 不能代替 Local Repair。


In [29]:
def make_fine_candidates(
    coarse_ranked,
    route_orders,
    existing_subsets,
    branch,
):
    """
    只在当前 Pareto anchors 周围补 ±FINE_STEP。
    这里仍然只改变 Depth，不改变 ranking membership rule。
    """
    if coarse_ranked.empty:
        return {}, pd.DataFrame()

    anchors = (
        coarse_ranked
        .loc[
            coarse_ranked[
                "pareto_front"
            ]
        ]
        .copy()
    )

    new_subsets = {}
    rows = []

    for row in anchors.itertuples(
        index=False
    ):
        route = row.route
        base_k = int(
            row.feature_count
        )

        for k in [
            base_k - FINE_STEP,
            base_k + FINE_STEP,
        ]:
            if not (
                MIN_SEARCH_SIZE
                <= k
                <= MAX_SEARCH_SIZE
            ):
                continue

            if route not in route_orders:
                continue

            order = route_orders[
                route
            ]

            actual_k = min(
                int(k),
                len(order),
            )

            name = (
                f"{branch}__{route}"
                f"__n{actual_k}"
            )

            if (
                name in existing_subsets
                or name in new_subsets
            ):
                continue

            new_subsets[
                name
            ] = list(
                order[
                    :actual_k
                ]
            )

            rows.append({
                "subset": name,
                "branch": branch,
                "route": route,
                "feature_count": (
                    actual_k
                ),
            })

    return (
        new_subsets,
        pd.DataFrame(
            rows
        ),
    )


def combine_and_rank(
    *ranked_tables,
):
    """
    合并任意多个已评估 candidate tables，
    去掉旧 Pareto 标记后重新计算整个搜索面的 Pareto rank。
    """
    parts = []

    for table in ranked_tables:
        if (
            isinstance(
                table,
                pd.DataFrame,
            )
            and not table.empty
        ):
            parts.append(
                table.drop(
                    columns=[
                        "pareto_rank",
                        "pareto_front",
                    ],
                    errors="ignore",
                )
            )

    if not parts:
        return pd.DataFrame()

    combined = (
        pd.concat(
            parts,
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["subset"]
        )
        .reset_index(
            drop=True
        )
    )

    return pareto_rank(
        combined
    )


def run_boundary_rounds(
    branch,
    initial_ranked,
    route_orders,
    subset_dict,
):
    """
    如果 Pareto front 贴着 coarse 边界，
    逐轮向外扩展，最多 MAX_BOUNDARY_ROUNDS。
    """
    if initial_ranked.empty:
        return (
            pd.DataFrame(),
            initial_ranked.copy(),
        )

    current_ranked = (
        initial_ranked.copy()
    )

    boundary_parts = []

    for round_no in range(
        1,
        MAX_BOUNDARY_ROUNDS + 1,
    ):
        (
            new_subsets,
            boundary_catalog,
        ) = boundary_candidates(
            current_ranked,
            route_orders,
            subset_dict,
            branch,
        )

        if boundary_catalog.empty:
            print(
                f"{branch} boundary round "
                f"{round_no}: no new proposal"
            )
            break

        boundary_catalog = (
            boundary_catalog
            .copy()
        )

        boundary_catalog[
            "boundary_round"
        ] = round_no

        subset_dict.update(
            new_subsets
        )

        print(
            f"{branch} boundary round "
            f"{round_no}"
        )

        evaluated, _ = run_stage1(
            branch,
            boundary_catalog,
            subset_dict,
        )

        boundary_parts.append(
            evaluated
        )

        current_ranked = (
            combine_and_rank(
                current_ranked,
                evaluated,
            )
        )

    boundary_ranked = (
        combine_and_rank(
            *boundary_parts
        )
        if boundary_parts
        else pd.DataFrame()
    )

    return (
        boundary_ranked,
        current_ranked,
    )


# ------------------------------------------------------------
# Boundary expansion
# ------------------------------------------------------------

tree_boundary_ranked = pd.DataFrame()
lr_boundary_ranked = pd.DataFrame()

tree_pre_fine_ranked = (
    tree_coarse_ranked.copy()
)

lr_pre_fine_ranked = (
    lr_coarse_ranked.copy()
)


if RUN_BOUNDARY_EXPANSION:

    if not tree_coarse_ranked.empty:
        (
            tree_boundary_ranked,
            tree_pre_fine_ranked,
        ) = run_boundary_rounds(
            "tree",
            tree_coarse_ranked,
            tree_routes,
            tree_subsets,
        )

    if not lr_coarse_ranked.empty:
        (
            lr_boundary_ranked,
            lr_pre_fine_ranked,
        ) = run_boundary_rounds(
            "lr",
            lr_coarse_ranked,
            lr_routes,
            lr_subsets,
        )


# ------------------------------------------------------------
# Fine depth search
# ------------------------------------------------------------

tree_fine_ranked = pd.DataFrame()
lr_fine_ranked = pd.DataFrame()

tree_stage1_final = (
    tree_pre_fine_ranked.copy()
)

lr_stage1_final = (
    lr_pre_fine_ranked.copy()
)


if RUN_FINE_SEARCH:

    if not tree_pre_fine_ranked.empty:

        (
            new_subsets,
            fine_catalog,
        ) = make_fine_candidates(
            tree_pre_fine_ranked,
            tree_routes,
            tree_subsets,
            "tree",
        )

        tree_subsets.update(
            new_subsets
        )

        if not fine_catalog.empty:

            tree_fine_ranked, _ = (
                run_stage1(
                    "tree",
                    fine_catalog,
                    tree_subsets,
                )
            )

        tree_stage1_final = (
            combine_and_rank(
                tree_pre_fine_ranked,
                tree_fine_ranked,
            )
        )


    if not lr_pre_fine_ranked.empty:

        (
            new_subsets,
            fine_catalog,
        ) = make_fine_candidates(
            lr_pre_fine_ranked,
            lr_routes,
            lr_subsets,
            "lr",
        )

        lr_subsets.update(
            new_subsets
        )

        if not fine_catalog.empty:

            lr_fine_ranked, _ = (
                run_stage1(
                    "lr",
                    fine_catalog,
                    lr_subsets,
                )
            )

        lr_stage1_final = (
            combine_and_rank(
                lr_pre_fine_ranked,
                lr_fine_ranked,
            )
        )


if not tree_stage1_final.empty:
    print(
        "Tree after boundary/fine search"
    )
    display(
        tree_stage1_final
        .sort_values(
            [
                "pareto_rank",
                "feature_count",
            ]
        )
    )

if not lr_stage1_final.empty:
    print(
        "LR after boundary/fine search"
    )
    display(
        lr_stage1_final
        .sort_values(
            [
                "pareto_rank",
                "feature_count",
            ]
        )
    )


tree boundary round 1
tree Stage-1: tree__relevance__boundary_n900 | 900 features
  checkpoint hit: tree__relevance__boundary_n900
tree Stage-1: tree__model_gain__boundary_n100 | 100 features
  checkpoint hit: tree__model_gain__boundary_n100
tree Stage-1: tree__temporal_relevance__boundary_n100 | 100 features
  checkpoint hit: tree__temporal_relevance__boundary_n100
tree Stage-1: tree__redundancy_aware__boundary_n100 | 100 features
  checkpoint hit: tree__redundancy_aware__boundary_n100
tree Stage-1: tree__redundancy_aware__boundary_n900 | 900 features
  checkpoint hit: tree__redundancy_aware__boundary_n900
tree Stage-1: tree__consensus__boundary_n100 | 100 features
  checkpoint hit: tree__consensus__boundary_n100
tree Stage-1: tree__consensus__boundary_n900 | 900 features
  checkpoint hit: tree__consensus__boundary_n900
tree boundary round 2: no new proposal
lr boundary round 1
lr Stage-1: lr__relevance__boundary_n100 | 100 features
  checkpoint hit: lr__relevance__boundary_n100
lr St

,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
43,tree__model_gain__boundary_n100,tree,100,0.840295,0.680590,90.909067,0.680893,-0.000638,0.000638,0.018916,0.615255,model_gain,1,True
45,tree__redundancy_aware__boundary_n100,tree,100,0.809864,0.619729,75.532008,0.620392,-0.000142,0.000142,0.019159,0.598341,redundancy_aware,1,True
47,tree__consensus__boundary_n100,tree,100,0.802995,0.605991,75.311812,0.606705,0.000965,0.000000,0.018627,0.597391,consensus,1,True
53,tree__model_gain__n150,tree,150,0.842090,0.684180,98.324924,0.684247,-0.001239,0.001239,0.019021,0.565693,model_gain,1,True
64,tree__redundancy_aware__n150,tree,150,0.811061,0.622122,75.291494,0.622606,-0.001042,0.001042,0.017314,0.522237,redundancy_aware,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,tree__consensus__boundary_n900,tree,900,0.843471,0.686942,212.670036,0.687043,-0.001151,0.001151,0.018711,0.576368,consensus,3,False
1,tree__relevance__n300,tree,300,0.814321,0.628643,102.916293,0.629260,-0.001054,0.001054,0.019039,0.526957,relevance,4,False
39,tree__consensus__n600,tree,600,0.841413,0.682825,164.033744,0.682791,-0.001839,0.001839,0.017324,0.512323,consensus,4,False
69,tree__redundancy_aware__n650,tree,650,0.841805,0.683611,170.456537,0.683511,-0.001790,0.001790,0.019354,0.516325,redundancy_aware,4,False


LR after boundary/fine search


,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
32,lr__redundancy_aware__boundary_n100,lr,100,0.759466,0.518931,111.102516,0.518781,0.000249,0.000000,0.021814,0.507874,redundancy_aware,1,True
46,lr__redundancy_aware__n150,lr,150,0.736854,0.473707,119.786221,0.473088,-0.004660,0.004660,0.020445,0.052814,redundancy_aware,1,True
14,lr__redundancy_aware__n200,lr,200,0.718389,0.436778,160.563415,0.435911,-0.007642,0.007642,0.016162,-0.244706,redundancy_aware,1,True
21,lr__lr_consensus__n200,lr,200,0.716776,0.433552,155.946993,0.432416,-0.004711,0.004711,0.017033,0.009298,lr_consensus,1,True
47,lr__redundancy_aware__n250,lr,250,0.728839,0.457678,166.736553,0.456068,-0.012961,0.012961,0.014472,-0.691717,redundancy_aware,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,lr__lr_consensus__n700,lr,700,0.713542,0.427084,443.495142,0.424933,-0.012286,0.012286,0.013860,-0.663209,lr_consensus,3,False
44,lr__temporal_relevance__n750,lr,750,0.685237,0.370474,408.926156,0.367621,-0.009910,0.009910,0.018536,-0.513766,temporal_relevance,3,False
45,lr__temporal_relevance__n850,lr,850,0.699626,0.399251,474.497491,0.396701,-0.009407,0.009407,0.018615,-0.440392,temporal_relevance,3,False
61,lr__lr_consensus__n850,lr,850,0.712102,0.424204,481.543273,0.422141,-0.011724,0.011724,0.014599,-0.616884,lr_consensus,3,False


## 9.5 Relevance Weighting Sensitivity Follow-up

这一步专门把 Notebook 9 的真实 sensitivity 结果接进 Notebook 10。

NB9 的诊断结果表明：

```text
Relevance 50/50
vs 60/40 / 40/60
```

在浅层 Top-k 上确实会换掉一批 feature，但没有必要把所有 weighting variants 都当成新的完整 Route，从 200 一直搜到 800。

所以这里采用一个更省、也更干净的策略：

> **先让 base Relevance Route 参加正常 Coarse / Boundary / Fine Search。**

只有当它进入当前 Pareto front，才在**相同的 competitive depth**上测试：

```text
60/40
40/60
```

### 简单例子

假设 base Relevance 的 Pareto candidate 是：

```text
50/50, Top500
```

那么这里不会再跑：

```text
60/40 × Top200/300/400/500/600/700/800
40/60 × Top200/300/400/500/600/700/800
```

而只补：

```text
60/40, Top500
40/60, Top500
```

这样回答的是一个非常具体的问题：

> **在同样 500 个 feature 的预算下，改变 Relevance 的合理解释，是否得到更好的 subset？**

这就是 NB9 的 Ranking Robustness 和 NB10 的 Wrapper Evaluation 真正连接起来的位置。

注意：这些 variant ranking 必须已经由 NB9 保存。NB10 不重新制造它们。


In [30]:
# ============================================================
# Optional NB9-saved Relevance variant follow-up
# ============================================================

def build_relevance_variant_followups(
    stage_ranked,
    sensitivity_orders,
    branch,
    variants,
):
    """
    只在 base relevance route 的 Pareto depths 上，
    比较 NB9 已经保存的 nearby weighting variants。
    """
    if (
        stage_ranked.empty
        or not sensitivity_orders
    ):
        return (
            {},
            pd.DataFrame(),
        )

    anchors = (
        stage_ranked
        .loc[
            (
                stage_ranked[
                    "pareto_front"
                ]
            )
            &
            (
                stage_ranked[
                    "route"
                ]
                .eq(
                    "relevance"
                )
            )
        ]
        .copy()
    )

    if anchors.empty:
        return (
            {},
            pd.DataFrame(),
        )

    new_subsets = {}
    rows = []

    anchor_sizes = sorted(
        anchors[
            "feature_count"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    for variant in variants:

        if variant not in (
            sensitivity_orders
        ):
            continue

        order = sensitivity_orders[
            variant
        ]

        for k in anchor_sizes:

            actual_k = min(
                int(k),
                len(order),
            )

            route_name = (
                f"relevance_{variant}"
            )

            name = (
                f"{branch}__{route_name}"
                f"__n{actual_k}"
            )

            new_subsets[
                name
            ] = list(
                order[
                    :actual_k
                ]
            )

            rows.append({
                "subset": name,
                "branch": branch,
                "route": route_name,
                "feature_count": (
                    actual_k
                ),
                "reason": (
                    "NB9_relevance_sensitivity_"
                    "followup_at_competitive_depth"
                ),
            })

    return (
        new_subsets,
        pd.DataFrame(
            rows
        ),
    )


tree_relevance_variant_ranked = (
    pd.DataFrame()
)

lr_relevance_variant_ranked = (
    pd.DataFrame()
)

tree_search_routes = dict(
    tree_routes
)

lr_search_routes = dict(
    lr_routes
)

tree_search_ranked = (
    tree_stage1_final.copy()
)

lr_search_ranked = (
    lr_stage1_final.copy()
)


if (
    RUN_RELEVANCE_VARIANT_FOLLOWUP
    and relevance_sensitivity_orders
):

    # The same NB9 ranking variants can be evaluated
    # by Tree and LR, but only if their base Relevance
    # route is currently competitive.
    (
        tree_variant_subsets,
        tree_variant_catalog,
    ) = build_relevance_variant_followups(
        tree_stage1_final,
        relevance_sensitivity_orders,
        "tree",
        RELEVANCE_VARIANTS_TO_TEST,
    )

    (
        lr_variant_subsets,
        lr_variant_catalog,
    ) = build_relevance_variant_followups(
        lr_stage1_final,
        relevance_sensitivity_orders,
        "lr",
        RELEVANCE_VARIANTS_TO_TEST,
    )


    if not tree_variant_catalog.empty:

        tree_subsets.update(
            tree_variant_subsets
        )

        for variant in (
            RELEVANCE_VARIANTS_TO_TEST
        ):
            if variant in (
                relevance_sensitivity_orders
            ):
                tree_search_routes[
                    f"relevance_{variant}"
                ] = (
                    relevance_sensitivity_orders[
                        variant
                    ]
                )

        (
            tree_relevance_variant_ranked,
            _,
        ) = run_stage1(
            "tree",
            tree_variant_catalog,
            tree_subsets,
        )

        tree_search_ranked = (
            combine_and_rank(
                tree_stage1_final,
                tree_relevance_variant_ranked,
            )
        )


    if not lr_variant_catalog.empty:

        lr_subsets.update(
            lr_variant_subsets
        )

        for variant in (
            RELEVANCE_VARIANTS_TO_TEST
        ):
            if variant in (
                relevance_sensitivity_orders
            ):
                lr_search_routes[
                    f"relevance_{variant}"
                ] = (
                    relevance_sensitivity_orders[
                        variant
                    ]
                )

        (
            lr_relevance_variant_ranked,
            _,
        ) = run_stage1(
            "lr",
            lr_variant_catalog,
            lr_subsets,
        )

        lr_search_ranked = (
            combine_and_rank(
                lr_stage1_final,
                lr_relevance_variant_ranked,
            )
        )


print(
    "Tree relevance-variant follow-up:"
)
display(
    tree_relevance_variant_ranked
)

print(
    "LR relevance-variant follow-up:"
)
display(
    lr_relevance_variant_ranked
)


tree Stage-1: tree__relevance_iv60_mi40__n500 | 500 features
  checkpoint hit: tree__relevance_iv60_mi40__n500
tree Stage-1: tree__relevance_iv60_mi40__n800 | 800 features
  checkpoint hit: tree__relevance_iv60_mi40__n800
tree Stage-1: tree__relevance_iv40_mi60__n500 | 500 features
  checkpoint hit: tree__relevance_iv40_mi60__n500
tree Stage-1: tree__relevance_iv40_mi60__n800 | 800 features
  checkpoint hit: tree__relevance_iv40_mi60__n800
lr Stage-1: lr__relevance_iv60_mi40__n600 | 600 features
  checkpoint hit: lr__relevance_iv60_mi40__n600
lr Stage-1: lr__relevance_iv60_mi40__n800 | 800 features
  checkpoint hit: lr__relevance_iv60_mi40__n800
lr Stage-1: lr__relevance_iv60_mi40__n850 | 850 features
  checkpoint hit: lr__relevance_iv60_mi40__n850
lr Stage-1: lr__relevance_iv40_mi60__n600 | 600 features
  checkpoint hit: lr__relevance_iv40_mi60__n600
lr Stage-1: lr__relevance_iv40_mi60__n800 | 800 features
  checkpoint hit: lr__relevance_iv40_mi60__n800
lr Stage-1: lr__relevance_iv40_

,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
0,tree__relevance_iv60_mi40__n500,tree,500,0.841188,0.682376,126.749269,0.682379,-0.001657,0.001657,0.017575,0.527776,relevance_iv60_mi40,1,True
1,tree__relevance_iv60_mi40__n800,tree,800,0.842080,0.684160,179.997510,0.684008,-0.001509,0.001509,0.016262,0.543062,relevance_iv60_mi40,1,True
2,tree__relevance_iv40_mi60__n500,tree,500,0.826086,0.652172,115.884652,0.652531,-0.001806,0.001806,0.018502,0.484323,relevance_iv40_mi60,2,False
3,tree__relevance_iv40_mi60__n800,tree,800,0.843264,0.686527,174.790850,0.686574,-0.001041,0.001041,0.019068,0.585455,relevance_iv40_mi60,1,True


LR relevance-variant follow-up:


,subset,branch,feature_count,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,route,pareto_rank,pareto_front
0,lr__relevance_iv60_mi40__n600,lr,600,0.708858,0.417715,328.019868,0.415360,-0.014181,0.014181,0.014197,-0.839654,relevance_iv60_mi40,1,True
1,lr__relevance_iv60_mi40__n800,lr,800,0.715696,0.431393,433.708116,0.429512,-0.010325,0.010325,0.016746,-0.487487,relevance_iv60_mi40,1,True
2,lr__relevance_iv60_mi40__n850,lr,850,0.712967,0.425933,452.178553,0.424074,-0.010337,0.010337,0.017205,-0.494150,relevance_iv60_mi40,2,False
3,lr__relevance_iv40_mi60__n600,lr,600,0.720997,0.441994,325.879021,0.439880,-0.013178,0.013178,0.015970,-0.727749,relevance_iv40_mi60,1,True
4,lr__relevance_iv40_mi60__n800,lr,800,0.719587,0.439175,430.052335,0.437265,-0.010993,0.010993,0.014293,-0.537289,relevance_iv40_mi60,1,True
5,lr__relevance_iv40_mi60__n850,lr,850,0.717091,0.434183,455.819812,0.432245,-0.010418,0.010418,0.014472,-0.491759,relevance_iv40_mi60,1,True


In [ ]:
def plot_size_search(result, title):
    if result.empty:
        return
    plt.figure(figsize=(9, 5))
    for route, group in result.groupby("route"):
        group = group.sort_values("feature_count")
        plt.plot(
            group["feature_count"],
            group["mean_weekly_gini"],
            marker="o",
            label=route,
        )
    front = result.loc[result["pareto_front"]]
    if not front.empty:
        plt.scatter(
            front["feature_count"],
            front["mean_weekly_gini"],
            marker="x",
            s=90,
            label="Pareto front",
        )
    plt.xlabel("Raw feature count")
    plt.ylabel("Mean weekly Gini")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_size_search(tree_search_ranked, "Tree: feature-count search")
plot_size_search(lr_search_ranked, "Logistic Regression: feature-count search")


## 10. Stage 3 — Restricted Local Repair

到这里，本阶段已完成：

```text
NB9
→ Direction

NB10 Stage 1–2
→ Depth

NB10 9.5
→ 只在 Relevance base route 真正有竞争力时
   补少量 NB9 已保存的 weighting variants
```

现在剩下最后一种 Top-k 本身修不了的问题：

> **固定 ranking 的某些成员，可能局部排错了。**

---

### 一个最简单的例子

假设当前最有竞争力的是：

```text
Top5 = {A, B, C, D, E}
```

但 Evidence 告诉我们：

```text
D、E
→ 属于同一个 redundancy cluster

F
→ 刚好在边界外
→ 另一条 route 排得很前
→ temporal 也稳定
```

那么只继续搜：

```text
Top4
Top5
Top6
```

永远不会直接测试：

```text
{A, B, C, D, F}
```

所以这里允许一个很小的：

```text
1-for-1 swap
```

---

### Local Repair 的算法怎么限制搜索爆炸？

不是：

```text
Parent 里面所有 feature
×
外面所有 feature
```

全部组合。

而是先构造两个很小的 pool。

**Drop Pool** 优先看：

```text
同 cluster 重复
→ 先考虑

temporal unstable
→ 再考虑

在 parent route 中靠近当前边界
→ 再考虑

relevance / gain 相对弱
→ 再考虑
```

**Add Pool** 优先看：

```text
parent route 边界外紧邻
+
其他 route 也排得靠前
+
temporal stable
+
relevance / gain 较强
```

然后才从：

```text
少量 Drop Pool
×
少量 Add Pool
```

中生成有限 proposal。

---

### NB9 的 Gain / Permutation disagreement 在这里怎么用？

NB9 的诊断结果表明：

> Gain 和 Permutation 在共同覆盖区域并不完全一致。

但 Permutation 只覆盖模型前部，所以不应该硬造一条完整 Permutation Route。

更合适的用途就是这里：

> **把 Permutation 当作 Local Repair 的附加线索，而不是新的全局 Direction。**

例如一个边界外 feature：

```text
Gain 一般
但 Permutation 很高
又被其他 Route 支持
```

它可以更值得进入 Add Pool。

这样 NB9 的 diagnostic 没有浪费，但也不会把不完整 Evidence 强行升级成完整 Route。

---

### 为什么叫 Restricted？

因为 Repair 的职责只是：

> **修 nesting error。**

它不是重新做一次全组合 feature selection。

因此限制：

- parent 数量；
- drop pool；
- add pool；
- 每个 parent proposal 数；
- repair rounds。

只有当前 Pareto parents 才值得支付这笔成本。


In [31]:
# ============================================================
# Stage 3 — Restricted Local Repair
# ============================================================

def choose_local_repair_parents(
    stage_ranked,
    budget,
):
    """
    优先从当前 Pareto front 选少量 parent。
    如果 front > budget，用 crowding distance
    保留不同 trade-off 区域。
    """
    if stage_ranked.empty:
        return []

    front = (
        stage_ranked
        .loc[
            stage_ranked[
                "pareto_front"
            ]
        ]
        .copy()
    )

    if len(front) <= budget:
        return front[
            "subset"
        ].tolist()

    distance = (
        crowding_distance(
            front
        )
    )

    chosen_index = (
        distance
        .sort_values(
            ascending=False
        )
        .head(
            budget
        )
        .index
    )

    return front.loc[
        chosen_index,
        "subset",
    ].tolist()


evidence_index = (
    selection_table
    .set_index("feature")
)


def _evidence_value(
    feature,
    column,
    default=np.nan,
):
    if (
        column not in evidence_index.columns
        or feature not in evidence_index.index
    ):
        return default

    value = evidence_index.at[
        feature,
        column,
    ]

    return (
        default
        if pd.isna(value)
        else value
    )


def _route_maps(
    route_orders,
):
    return {
        route_name: {
            feature: rank
            for rank, feature
            in enumerate(
                order,
                start=1,
            )
        }
        for route_name, order
        in route_orders.items()
    }


def build_local_repair_proposals(
    branch,
    stage_ranked,
    subset_dict,
    route_orders,
    round_no,
    existing_memberships,
):
    """
    Local Repair 只做有理由的 1-for-1 swap。

    Drop side:
    - 同 numerical cluster 在 parent 中重复出现；
    - temporal 不稳定；
    - 在 parent route 中靠近边界；
    - relevance / gain 较弱。

    Add side:
    - parent route 边界外紧邻；
    - 其他 route 在前部支持；
    - temporal stable；
    - relevance / gain 较强。

    不做完整 drop × add 笛卡尔积，只截取少量 proposal。
    """
    if stage_ranked.empty:
        return (
            {},
            pd.DataFrame(),
        )

    parent_names = (
        choose_local_repair_parents(
            stage_ranked,
            LOCAL_REPAIR_PARENT_BUDGET,
        )
    )

    if not parent_names:
        return (
            {},
            pd.DataFrame(),
        )

    route_maps = (
        _route_maps(
            route_orders
        )
    )

    stage_lookup = (
        stage_ranked
        .drop_duplicates(
            subset=["subset"]
        )
        .set_index("subset")
    )

    proposals = {}
    rows = []

    proposal_id = 0

    for parent_index, parent_name in enumerate(
        parent_names,
        start=1,
    ):
        if (
            parent_name not in subset_dict
            or parent_name not in stage_lookup.index
        ):
            continue

        parent_row = (
            stage_lookup.loc[
                parent_name
            ]
        )

        route = str(
            parent_row[
                "route"
            ]
        )

        if route not in route_orders:
            continue

        current = list(
            subset_dict[
                parent_name
            ]
        )

        current_set = set(
            current
        )

        k = len(
            current
        )

        base_rank_map = (
            route_maps[
                route
            ]
        )

        # ----------------------------------------------------
        # Drop pool
        # ----------------------------------------------------

        numeric_groups = {}

        if (
            "numeric_redundancy_group"
            in evidence_index.columns
        ):
            for feature in current:
                group = _evidence_value(
                    feature,
                    "numeric_redundancy_group",
                    default=None,
                )

                if (
                    group is not None
                    and pd.notna(group)
                ):
                    numeric_groups.setdefault(
                        str(group),
                        [],
                    ).append(
                        feature
                    )

        duplicated_numeric_features = set()

        for members in (
            numeric_groups.values()
        ):
            if len(members) > 1:
                duplicated_numeric_features.update(
                    members
                )

        drop_rows = []

        for feature in current:

            route_rank = int(
                base_rank_map.get(
                    feature,
                    len(
                        route_orders[
                            route
                        ]
                    )
                    + 1,
                )
            )

            temporal_stable = bool(
                _evidence_value(
                    feature,
                    "temporal_stable_flag",
                    default=False,
                )
            )

            relevance = float(
                _evidence_value(
                    feature,
                    "relevance_score",
                    default=-np.inf,
                )
            )

            gain = float(
                _evidence_value(
                    feature,
                    "gain_percentile",
                    default=-np.inf,
                )
            )

            drop_rows.append({
                "feature": feature,
                "redundant_in_parent": (
                    feature
                    in duplicated_numeric_features
                ),
                "temporal_unstable": (
                    not temporal_stable
                ),
                "route_rank": route_rank,
                "relevance_score": relevance,
                "gain_percentile": gain,
            })

        drop_table = (
            pd.DataFrame(
                drop_rows
            )
            .sort_values(
                [
                    "redundant_in_parent",
                    "temporal_unstable",
                    "route_rank",
                    "relevance_score",
                    "gain_percentile",
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True,
                    True,
                ],
            )
            .head(
                LOCAL_REPAIR_DROP_POOL
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # Add pool
        # ----------------------------------------------------

        candidate_universe = set()

        # Parent route just outside current boundary.
        parent_order = (
            route_orders[
                route
            ]
        )

        boundary_outside = [
            feature
            for feature in parent_order
            if feature not in current_set
        ][
            : max(
                LOCAL_REPAIR_ADD_POOL * 4,
                40,
            )
        ]

        candidate_universe.update(
            boundary_outside
        )

        # Alternate routes can nominate features that the
        # parent route placed too far back.
        alternate_front_limit = min(
            len(candidate_features),
            k + 50,
        )

        for alt_route, order in (
            route_orders.items()
        ):
            candidate_universe.update(
                feature
                for feature in (
                    order[
                        :alternate_front_limit
                    ]
                )
                if feature
                not in current_set
            )

        add_rows = []

        for feature in candidate_universe:

            ranks = [
                rank_map.get(
                    feature,
                    len(candidate_features)
                    + 1,
                )
                for rank_map
                in route_maps.values()
            ]

            front_votes = sum(
                rank <= k
                for rank in ranks
            )

            best_rank = min(
                ranks
            )

            base_rank = (
                base_rank_map.get(
                    feature,
                    len(candidate_features)
                    + 1,
                )
            )

            temporal_stable = bool(
                _evidence_value(
                    feature,
                    "temporal_stable_flag",
                    default=False,
                )
            )

            relevance = float(
                _evidence_value(
                    feature,
                    "relevance_score",
                    default=-np.inf,
                )
            )

            gain = float(
                _evidence_value(
                    feature,
                    "gain_percentile",
                    default=-np.inf,
                )
            )

            permutation = float(
                _evidence_value(
                    feature,
                    "permutation_percentile",
                    default=-np.inf,
                )
            )

            add_rows.append({
                "feature": feature,
                "front_votes": (
                    front_votes
                ),
                "best_route_rank": (
                    best_rank
                ),
                "parent_route_rank": (
                    base_rank
                ),
                "temporal_stable": (
                    temporal_stable
                ),
                "relevance_score": (
                    relevance
                ),
                "gain_percentile": (
                    gain
                ),
                "permutation_percentile": (
                    permutation
                ),
            })

        add_table = (
            pd.DataFrame(
                add_rows
            )
        )

        if add_table.empty:
            continue

        add_table = (
            add_table
            .sort_values(
                [
                    "front_votes",
                    "best_route_rank",
                    "parent_route_rank",
                    "temporal_stable",
                    "relevance_score",
                    "gain_percentile",
                    "permutation_percentile",
                ],
                ascending=[
                    False,
                    True,
                    True,
                    False,
                    False,
                    False,
                    False,
                ],
            )
            .head(
                LOCAL_REPAIR_ADD_POOL
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # Small deterministic swap set
        # ----------------------------------------------------

        pair_rows = []

        for drop_position, drop_row in (
            drop_table.iterrows()
        ):
            for add_position, add_row in (
                add_table.iterrows()
            ):
                pair_rows.append({
                    "drop_feature": (
                        drop_row[
                            "feature"
                        ]
                    ),
                    "add_feature": (
                        add_row[
                            "feature"
                        ]
                    ),
                    "drop_position": int(
                        drop_position
                    ),
                    "add_position": int(
                        add_position
                    ),
                    "pair_priority": (
                        int(
                            drop_position
                        )
                        + int(
                            add_position
                        )
                    ),
                })

        pair_table = (
            pd.DataFrame(
                pair_rows
            )
            .sort_values(
                [
                    "pair_priority",
                    "drop_position",
                    "add_position",
                ]
            )
        )

        accepted = 0

        for pair in (
            pair_table
            .itertuples(
                index=False
            )
        ):
            child = [
                feature
                for feature in current
                if feature
                != pair.drop_feature
            ]

            child.append(
                pair.add_feature
            )

            membership = frozenset(
                child
            )

            if membership in existing_memberships:
                continue

            existing_memberships.add(
                membership
            )

            proposal_id += 1

            name = (
                f"{branch}__repair"
                f"_r{round_no}"
                f"_p{parent_index}"
                f"_s{proposal_id}"
            )

            proposals[
                name
            ] = child

            rows.append({
                "subset": name,
                "branch": branch,
                "route": route,
                "feature_count": (
                    len(child)
                ),
                "repair_round": (
                    round_no
                ),
                "parent_subset": (
                    parent_name
                ),
                "drop_feature": (
                    pair.drop_feature
                ),
                "add_feature": (
                    pair.add_feature
                ),
            })

            accepted += 1

            if accepted >= (
                LOCAL_REPAIR_PROPOSALS_PER_PARENT
            ):
                break

    return (
        proposals,
        pd.DataFrame(
            rows
        ),
    )


def run_local_repair_rounds(
    branch,
    initial_ranked,
    subset_dict,
    route_orders,
):
    if initial_ranked.empty:
        return (
            pd.DataFrame(),
            initial_ranked.copy(),
            pd.DataFrame(),
        )

    current_ranked = (
        initial_ranked.copy()
    )

    existing_memberships = {
        frozenset(
            features
        )
        for features
        in subset_dict.values()
    }

    repair_result_parts = []
    repair_catalog_parts = []

    for round_no in range(
        1,
        MAX_LOCAL_REPAIR_ROUNDS + 1,
    ):

        (
            proposals,
            repair_catalog,
        ) = build_local_repair_proposals(
            branch,
            current_ranked,
            subset_dict,
            route_orders,
            round_no,
            existing_memberships,
        )

        if repair_catalog.empty:
            print(
                f"{branch} repair round "
                f"{round_no}: no proposal"
            )
            break

        subset_dict.update(
            proposals
        )

        evaluated, _ = run_stage1(
            branch,
            repair_catalog,
            subset_dict,
        )

        repair_result_parts.append(
            evaluated
        )

        repair_catalog_parts.append(
            repair_catalog
        )

        current_ranked = (
            combine_and_rank(
                current_ranked,
                evaluated,
            )
        )

        print(
            f"{branch} after repair "
            f"round {round_no}"
        )

        display(
            current_ranked
            .sort_values(
                [
                    "pareto_rank",
                    "feature_count",
                ]
            )
            .head(30)
        )

    repair_ranked = (
        combine_and_rank(
            *repair_result_parts
        )
        if repair_result_parts
        else pd.DataFrame()
    )

    repair_catalog_all = (
        pd.concat(
            repair_catalog_parts,
            ignore_index=True,
        )
        if repair_catalog_parts
        else pd.DataFrame()
    )

    return (
        repair_ranked,
        current_ranked,
        repair_catalog_all,
    )


tree_repair_ranked = pd.DataFrame()
lr_repair_ranked = pd.DataFrame()

tree_repair_catalog = pd.DataFrame()
lr_repair_catalog = pd.DataFrame()

tree_search_final = (
    tree_search_ranked.copy()
)

lr_search_final = (
    lr_search_ranked.copy()
)


if RUN_LOCAL_REPAIR:

    if not tree_search_ranked.empty:

        (
            tree_repair_ranked,
            tree_search_final,
            tree_repair_catalog,
        ) = run_local_repair_rounds(
            "tree",
            tree_search_ranked,
            tree_subsets,
            tree_search_routes,
        )


    if not lr_search_ranked.empty:

        (
            lr_repair_ranked,
            lr_search_final,
            lr_repair_catalog,
        ) = run_local_repair_rounds(
            "lr",
            lr_search_ranked,
            lr_subsets,
            lr_search_routes,
        )


local_repair_plan = {
    "parent_budget": (
        LOCAL_REPAIR_PARENT_BUDGET
    ),
    "drop_pool_size": (
        LOCAL_REPAIR_DROP_POOL
    ),
    "add_pool_size": (
        LOCAL_REPAIR_ADD_POOL
    ),
    "proposals_per_parent": (
        LOCAL_REPAIR_PROPOSALS_PER_PARENT
    ),
    "max_rounds": (
        MAX_LOCAL_REPAIR_ROUNDS
    ),
    "status": (
        "implemented; runs only when "
        "RUN_LOCAL_REPAIR=True"
    ),
}


print(
    "Local repair plan:"
)

display(
    pd.DataFrame([
        local_repair_plan
    ])
)


Local repair plan:


,parent_budget,drop_pool_size,add_pool_size,proposals_per_parent,max_rounds,status
0,4,8,12,12,2,implemented; runs only when RUN_LOCAL_REPAIR=True


## 11. Stage 4 — F1–F4 时间折评估

Stage-1 只看：

```text
Train 0–55
→ Assess 56–63
```

它可能碰巧偏爱最近时期。

所以只有少量 finalists 才支付完整时间验证成本：

```text
F1  Train 0–31 → Assess 32–39
F2  Train 0–39 → Assess 40–47
F3  Train 0–47 → Assess 48–55
F4  Train 0–55 → Assess 56–63
```

每个 fold 都重新训练对应 learner：

```text
Tree finalist
→ LightGBM

LR finalist
→ Logistic Regression
```

这一步针对的问题是：

> **“最近窗口看起来很好”是否能重复到不同历史时间点？**

为什么不让所有 coarse candidates 都跑 F1–F4？

因为那会把最昂贵的 temporal evaluation 平均撒给大量明显没有竞争力的 subsets。

所以：

```text
Coarse / Fine / Repair
→ 便宜地淘汰

少量 finalists
→ 才跑 F1–F4
```


In [32]:
TEMPORAL_FOLDS = [
    ("F1", 0, 31, 32, 39),
    ("F2", 0, 39, 40, 47),
    ("F3", 0, 47, 48, 55),
    ("F4", 0, 55, 56, 63),
]


def choose_temporal_finalists(stage1_ranked, budget):
    if stage1_ranked.empty:
        return []
    front = stage1_ranked.loc[stage1_ranked["pareto_front"]].copy()
    if len(front) <= budget:
        return front["subset"].tolist()

    distance = crowding_distance(front)
    chosen_index = distance.sort_values(ascending=False).head(budget).index
    return front.loc[chosen_index, "subset"].tolist()


def evaluate_full_temporal_branch(branch, subset_names, subset_dict):
    summary_rows, fold_rows, weekly_parts = [], [], []

    for subset_name in subset_names:
        features = subset_dict[subset_name]
        prediction_parts = []
        this_fold_rows = []

        print(f"\n{branch} full temporal: {subset_name}")

        for fold_name, train_start, train_end, assess_start, assess_end in TEMPORAL_FOLDS:
            print(" ", fold_name)
            summary, weekly = evaluate_candidate(
                branch,
                subset_name,
                features,
                train_start,
                train_end,
                assess_start,
                assess_end,
            )

            # Fold-level Gini uses the complete 8-week assessment block.
            this_fold_rows.append({
                "subset": subset_name,
                "branch": branch,
                "fold": fold_name,
                "feature_count": len(features),
                "gini": summary["gini"],
            })

            weekly["subset"] = subset_name
            weekly["fold"] = fold_name
            weekly_parts.append(weekly)

        fold_df = pd.DataFrame(this_fold_rows)
        subset_weekly = pd.concat(
            [w for w in weekly_parts if w["subset"].iloc[0] == subset_name],
            ignore_index=True,
        )
        temporal = temporal_metrics(subset_weekly[[WEEK_COL, "gini", "sample_count"]])

        summary_rows.append({
            "subset": subset_name,
            "branch": branch,
            "route": (
                subset_route_registry.get(
                    subset_name,
                    "unknown",
                )
            ),
            "feature_count": len(features),
            "mean_fold_gini": float(fold_df["gini"].mean()),
            "sd_fold_gini": float(fold_df["gini"].std(ddof=1)),
            **temporal,
        })
        fold_rows.extend(this_fold_rows)

    summary = pd.DataFrame(summary_rows)
    ranked = pareto_rank(summary) if not summary.empty else pd.DataFrame()
    folds = pd.DataFrame(fold_rows)
    weekly = pd.concat(weekly_parts, ignore_index=True) if weekly_parts else pd.DataFrame()
    return ranked, folds, weekly


tree_final_ranked = pd.DataFrame()
lr_final_ranked = pd.DataFrame()
tree_fold_metrics = pd.DataFrame()
lr_fold_metrics = pd.DataFrame()

if RUN_FULL_TEMPORAL:
    tree_finalists = choose_temporal_finalists(
        tree_search_final, MAX_FULL_TEMPORAL_PER_BRANCH
    )
    lr_finalists = choose_temporal_finalists(
        lr_search_final, MAX_FULL_TEMPORAL_PER_BRANCH
    )

    print("Tree finalists:", tree_finalists)
    print("LR finalists:", lr_finalists)

    if tree_finalists:
        tree_final_ranked, tree_fold_metrics, tree_final_weekly = (
            evaluate_full_temporal_branch("tree", tree_finalists, tree_subsets)
        )
        display(tree_final_ranked.sort_values(["pareto_rank", "feature_count"]))

    if lr_finalists:
        lr_final_ranked, lr_fold_metrics, lr_final_weekly = (
            evaluate_full_temporal_branch("lr", lr_finalists, lr_subsets)
        )
        display(lr_final_ranked.sort_values(["pareto_rank", "feature_count"]))


Tree finalists: ['tree__relevance__n500', 'tree__temporal_relevance__n200', 'tree__redundancy_aware__n300', 'tree__redundancy_aware__boundary_n100']
LR finalists: ['lr__relevance__n600', 'lr__redundancy_aware__boundary_n100', 'lr__lr_consensus__n500', 'lr__redundancy_aware__boundary_n1000']

tree full temporal: tree__relevance__n500
  F1
  checkpoint hit: tree__relevance__n500
  F2
  checkpoint hit: tree__relevance__n500
  F3
  checkpoint hit: tree__relevance__n500
  F4
  checkpoint hit: tree__relevance__n500

tree full temporal: tree__temporal_relevance__n200
  F1
  checkpoint hit: tree__temporal_relevance__n200
  F2
  checkpoint hit: tree__temporal_relevance__n200
  F3
  checkpoint hit: tree__temporal_relevance__n200
  F4
  checkpoint hit: tree__temporal_relevance__n200

tree full temporal: tree__redundancy_aware__n300
  F1
  checkpoint hit: tree__redundancy_aware__n300
  F2
  checkpoint hit: tree__redundancy_aware__n300
  F3
  checkpoint hit: tree__redundancy_aware__n300
  F4
  chec

,subset,branch,route,feature_count,mean_fold_gini,sd_fold_gini,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,pareto_rank,pareto_front
3,tree__redundancy_aware__boundary_n100,tree,redundancy_aware,100,0.624445,0.009002,0.623004,0.000084,0.000000,0.027667,0.609170,1,True
1,tree__temporal_relevance__n200,tree,temporal_relevance,200,0.641696,0.007116,0.640732,-0.000666,0.000666,0.021920,0.571208,1,True
2,tree__redundancy_aware__n300,tree,redundancy_aware,300,0.666806,0.010443,0.665165,0.000728,0.000000,0.023062,0.653634,1,True
0,tree__relevance__n500,tree,relevance,500,0.672974,0.010747,0.671295,0.000649,0.000000,0.023182,0.659704,1,True



lr full temporal: lr__relevance__n600
  F1
  checkpoint hit: lr__relevance__n600
  F2
  checkpoint hit: lr__relevance__n600
  F3
  checkpoint hit: lr__relevance__n600
  F4
  checkpoint hit: lr__relevance__n600

lr full temporal: lr__redundancy_aware__boundary_n100
  F1
  checkpoint hit: lr__redundancy_aware__boundary_n100
  F2
  checkpoint hit: lr__redundancy_aware__boundary_n100
  F3
  checkpoint hit: lr__redundancy_aware__boundary_n100
  F4
  checkpoint hit: lr__redundancy_aware__boundary_n100

lr full temporal: lr__lr_consensus__n500
  F1
  checkpoint hit: lr__lr_consensus__n500
  F2
  checkpoint hit: lr__lr_consensus__n500
  F3
  checkpoint hit: lr__lr_consensus__n500
  F4
  checkpoint hit: lr__lr_consensus__n500

lr full temporal: lr__redundancy_aware__boundary_n1000
  F1
  checkpoint hit: lr__redundancy_aware__boundary_n1000
  F2
  checkpoint hit: lr__redundancy_aware__boundary_n1000
  F3
  checkpoint hit: lr__redundancy_aware__boundary_n1000
  F4
  checkpoint hit: lr__redundanc

,subset,branch,route,feature_count,mean_fold_gini,sd_fold_gini,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,pareto_rank,pareto_front
1,lr__redundancy_aware__boundary_n100,lr,redundancy_aware,100,0.461670,0.078683,0.471174,0.003647,0.000000,0.066556,0.437896,1,True
2,lr__lr_consensus__n500,lr,lr_consensus,500,0.420194,0.035872,0.419981,-0.004080,0.004080,0.030466,0.045721,1,True
0,lr__relevance__n600,lr,relevance,600,0.438849,0.045696,0.446201,0.000483,0.000000,0.057612,0.417395,1,True
3,lr__redundancy_aware__boundary_n1000,lr,redundancy_aware,1000,0.449704,0.015791,0.453637,-0.001889,0.001889,0.042073,0.266333,1,True


## 12. Final Pareto 与 1-SE-style Plateau

Final Pareto 可能仍然有多个方案。

这是正常的，因为：

```text
Prediction
Temporal Robustness
Feature Count
```

本来就可能互相冲突。

本 Notebook额外给一个：

> **1-SE-style plateau recommendation**

但它只是 practical tolerance，不是正式统计 CI。

步骤：

```text
1. 只看 Final Pareto Front
2. 找 mean fold Gini 最高的 candidate
3. 用 4-fold SD / sqrt(4) 构造一个简单 tolerance
4. 在没有明显差于最好结果的 candidates 中
   优先选择 feature 更少的
5. 再比较 slope / residual SD
```

它针对的问题是：

> **避免因为一个 noisy 单点比邻居高 0.001，就宣称它是真实唯一最优。**


In [ ]:
def plateau_recommendation(final_ranked):
    if final_ranked.empty:
        return None

    front = final_ranked.loc[final_ranked["pareto_front"]].copy()
    best = front.loc[front["mean_fold_gini"].idxmax()]

    se_like = float(best["sd_fold_gini"] / np.sqrt(4))
    threshold = float(best["mean_fold_gini"] - se_like)

    eligible = front.loc[
        front["mean_fold_gini"].ge(threshold)
    ].copy()

    eligible = eligible.sort_values(
        [
            "feature_count",
            "negative_slope",
            "weekly_residual_sd",
            "mean_fold_gini",
        ],
        ascending=[True, True, True, False],
    )

    chosen = eligible.iloc[0]
    return {
        "subset": chosen["subset"],
        "route": (
            chosen["route"]
            if "route" in chosen.index
            else "unknown"
        ),
        "feature_count": int(chosen["feature_count"]),
        "mean_fold_gini": float(chosen["mean_fold_gini"]),
        "best_mean_fold_gini": float(best["mean_fold_gini"]),
        "plateau_threshold": threshold,
        "note": "1-SE-style plateau rule; not a formal CI because temporal folds are dependent.",
    }


tree_recommendation = plateau_recommendation(tree_final_ranked)
lr_recommendation = plateau_recommendation(lr_final_ranked)

print("Tree recommendation:", tree_recommendation)
print("LR recommendation:", lr_recommendation)


## 13. 特征冻结前的鲁棒性检查

冻结以前，现在要看两层 Robustness。

第一层是 Notebook 10 自己的：

```text
相邻 k 是否形成 plateau？
不同 Route 是否在类似深度有竞争力？
Local Repair 是否只是小幅修正？
F1–F4 是否能重复？
```

第二层来自 Notebook 9：

> **如果最终赢家恰好来自一个 NB9 已经证明“定义敏感”的 Route，就不可忽略这个不确定性不存在。**

NB9 有两个特别明显的 guardrail。

### Interaction

NB9 的 exclusive rescue 数量：

```text
Top300 pairs → 38
Top500 pairs → 52
Top800 pairs → 71
```

300 对 800 的 rescue-set Jaccard 只有约：

```text
0.535
```

所以如果最终 Tree winner 来自：

```text
interaction_relevance
```

正确做法不是直接 Freeze。

而是：

> **回 NB9 导出至少一个更宽 pair-limit 的 Interaction ranking，再做一次 targeted follow-up。**

---

### Temporal

本次：

```text
547 / 1209
≈ 45.2%
```

features 的 `temporal_gini_min` 落在 ±0.01。

这说明 `0` 这个人工 boundary 附近非常拥挤。

所以如果最终 winner 来自：

```text
temporal_relevance
```

也应该在 Freeze 前检查：

> label-first ranking 是否对 boundary 过度敏感，是否需要 continuous temporal ranking 做一次 targeted comparison。

---

### Relevance

本次 60/40、40/60 相对 50/50 确实改变了一些 Top-k membership。

但 NB10 已经采取更便宜的处理：

```text
先跑 base Relevance
        ↓
只有 base Relevance 进入 Pareto
        ↓
才在相同 depth 测 60/40、40/60
```

因此不用一开始把整个搜索空间乘三。

---

这个 guardrail 的意义不是让 NB10 停下来。

而是：

> **允许 Search Execution 继续，但不允许在已知上游定义仍敏感时过早 Freeze。**

这正好把三本 Notebook 连起来：

```text
NB8
Evidence Robustness

NB9
Ranking Robustness

NB10
Solution Robustness
        ↓
三者都没有明显未解决问题
        ↓
Freeze
```


## 13.5 LR 来源变量与非线性变换候选池

到这里需要把 **Local Repair** 和 **Representation Rescue** 分开。

Stage 3 的 Local Repair 解决的是：

```text
某个 raw feature 本来就不错
但在当前 ranking 里排错了一点
        ↓
允许边界附近做少量 1-for-1 swap
```

它不能解决另一类盲点：

```text
raw X 对 Logistic Regression 很弱
        ↓
NB10 把 X 删掉
        ↓
下一本本来可能发现

log(X)
spline(X)
bin(X)

很有用
```

也不能完全解决：

```text
A 单独弱
B 单独弱
        ↓
A / B
A - B
A × B

却可能有价值
```

因此 **LR 在 NB10 的输出不应该解释成最终 design matrix**。

更准确的接口是：

```text
NB10
1209 candidate source features
        ↓
Wrapper Search
        ↓
LR Core Source Set
        +
Transformation Rescue Pool
        ↓
NB11
只在这个受控 source pool 上
做 log / bin / spline / ratio / interaction 等表示搜索
        ↓
再次 selection / regularization
        ↓
Final LR Design Matrix
```

---

### 为什么 rescue 是合理的？

高维 screening 的经典工作本来就是：

> 先用便宜规则把维度从极高压到中等，再把更精细的方法放到缩小后的空间。

Fan & Lv (2008) 的 Sure Independence Screening 还专门提出 iterative screening，
原因之一就是纯 marginal screening 可能遗漏“单独看不强、条件化之后才重要”的变量。

这里不直接实现 SIS / ISIS；我们借用的是同一个 **screen → revisit suspicious omissions → expensive search later** 的思想。

另外，Tree 可以作为 LR 的一种 **interaction / nonlinearity clue**。
Friedman & Popescu (2008) 的 RuleFit 就展示了：
可以利用 Tree 产生非线性规则和 interaction，再交给稀疏线性模型选择。

对于 interaction，本阶段不强制采用最严格的：

```text
只有 A、B 都是强 main effect
才允许 A × B
```

Hierarchical interaction 方法（Bien, Taylor & Tibshirani, 2013）
说明 hierarchy 是控制 interaction search space 的有用结构假设；
但这里把它当 **预算约束**，而不是把 strong hierarchy 当成永远正确的事实。

因此本 Notebook采用四个 rescue channel：

1. **Nonlinear disagreement**  
   MI 较强，但 IV / 当前 LR relevance 明显更弱；
2. **Tree–LR disagreement**  
   Tree evidence / Tree ranking 明显比 LR ranking 强；
3. **Interaction evidence**  
   NB8 已发现该 source feature 参与较强 tree-path interaction；
4. **Boundary disagreement**  
   没进最终 Core，但在其他 LR routes 的相邻前部反复出现。

这些 channel **不会在 NB10 直接生成 transformation**。

它们只回答：

> **哪些被 Core 排除的 source variables，值得在 NB11 再给一次表示搜索预算？**

---

### 一个示意例子

假设 NB10 最终 Core 有 400 个 source features。

```text
X1
LR raw rank = 780

但：
MI percentile = 0.96
Tree rank = 85
```

直接删除：

```text
X1 → 永久消失
```

风险太高。

Rescue 后：

```text
Core = 400
Rescue = 80（示例，不是固定答案）
        ↓
NB11 source pool ≈ 480
```

然后 NB11 才判断：

```text
X1
log(X1)
bin(X1)
spline(X1)
```

是否值得留下。

所以 Rescue 的目的不是“让删掉的变量都回来”，而是：

> **让少量有结构性理由被怀疑为 false negative 的 source feature 保留下一轮搜索资格。**

---

### 参考文献

- Fan, J. & Lv, J. (2008), *Sure Independence Screening for Ultrahigh Dimensional Feature Space*, JRSS-B, DOI: 10.1111/j.1467-9868.2008.00674.x.
- Friedman, J. H. & Popescu, B. E. (2008), *Predictive Learning via Rule Ensembles*, Annals of Applied Statistics, DOI: 10.1214/07-AOAS148.
- Bien, J., Taylor, J. & Tibshirani, R. (2013), *A Lasso for Hierarchical Interactions*, Annals of Statistics, DOI: 10.1214/13-AOS1096.

> **注意：下面的 40 / 30 / 0.75 等数值都是 search-budget knobs，不是文献给出的理论常数。**
> 它们以后可以根据 NB10 实际结果调整，但不应该用 Validation / OOT 去调。


In [ ]:
# ============================================================
# 13.5 LR Source-Space Rescue for Notebook 11
# ============================================================
#
# IMPORTANT:
# - This does NOT create transformed features in NB10.
# - This does NOT read Validation / OOT.
# - It only reserves a small set of raw/source variables
#   that have a structural reason to receive FE budget in NB11.


# ------------------------------------------------------------
# Rescue budgets / engineering guardrails
# ------------------------------------------------------------
LR_RESCUE_NONLINEAR_BUDGET = 40
LR_RESCUE_TREE_DISAGREEMENT_BUDGET = 40
LR_RESCUE_INTERACTION_BUDGET = 40
LR_RESCUE_BOUNDARY_BUDGET = 30

# Union of all rescue channels is capped again.
LR_RESCUE_MAX_TOTAL = 120

# Nonlinear disagreement:
# MI can capture more general dependence than a purely linear LR effect.
# These are screening heuristics, not theoretical cutoffs.
LR_RESCUE_MI_MIN = 0.75
LR_RESCUE_MI_IV_GAP_MIN = 0.10

# Tree–LR disagreement:
# allow a feature to re-enter when Tree evidence is strong or
# its Tree ranking is much better than its LR ranking.
LR_RESCUE_TREE_MODEL_MIN = 0.85
LR_RESCUE_TREE_RANK_GAP_MIN = 100
LR_RESCUE_TREE_FRONT_EXTRA = 100

# Boundary disagreement:
# look only a short distance beyond the final LR depth.
LR_RESCUE_BOUNDARY_MARGIN = 50


def _safe_numeric_column(
    frame,
    column,
):
    if column not in frame.columns:
        return pd.Series(
            np.nan,
            index=frame.index,
            dtype=float,
        )

    return pd.to_numeric(
        frame[column],
        errors="coerce",
    )


def _safe_bool_column(
    frame,
    column,
):
    if column not in frame.columns:
        return pd.Series(
            False,
            index=frame.index,
            dtype=bool,
        )

    series = frame[column]

    if pd.api.types.is_bool_dtype(
        series
    ):
        return (
            series
            .fillna(False)
            .astype(bool)
        )

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({
            "true",
            "1",
            "yes",
            "y",
        })
    )


def _route_rank_table(
    routes,
    features,
    prefix,
    front_k=None,
):
    rank_maps = {
        route_name: {
            feature: rank
            for rank, feature
            in enumerate(
                order,
                start=1,
            )
        }
        for route_name, order
        in routes.items()
    }

    rows = []

    fallback_rank = (
        len(candidate_features)
        + 1
    )

    for feature in features:

        ranks = [
            rank_map.get(
                feature,
                fallback_rank,
            )
            for rank_map
            in rank_maps.values()
        ]

        row = {
            "feature": feature,
            f"{prefix}_best_rank": int(
                min(ranks)
            ),
            f"{prefix}_median_rank": float(
                np.median(ranks)
            ),
        }

        if front_k is not None:
            row[
                f"{prefix}_front_votes_at_k"
            ] = int(
                sum(
                    rank <= front_k
                    for rank in ranks
                )
            )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


def _take_channel(
    frame,
    channel,
    budget,
    sort_columns,
    ascending,
    rationale,
):
    if frame.empty:
        return pd.DataFrame()

    chosen = (
        frame
        .sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last",
        )
        .head(
            int(budget)
        )
        .copy()
    )

    chosen[
        "rescue_channel"
    ] = channel

    chosen[
        "channel_rank"
    ] = np.arange(
        1,
        len(chosen) + 1,
    )

    chosen[
        "rationale"
    ] = rationale

    return chosen


def build_lr_source_rescue(
    lr_core_subset_name,
    tree_core_subset_name=None,
):
    """
    Build:
      1) LR Core source features
      2) evidence-guided rescue candidates
      3) combined source pool for NB11

    The rescue pool is deliberately conservative:
    it gives a small number of omitted raw variables
    permission to enter NB11 representation search.
    """

    if (
        lr_core_subset_name is None
        or lr_core_subset_name
        not in lr_subsets
    ):
        raise ValueError(
            "lr_core_subset_name 必须是 NB10 已评估的 LR subset。"
        )

    core_features = list(
        lr_subsets[
            lr_core_subset_name
        ]
    )

    core_set = set(
        core_features
    )

    k = len(
        core_features
    )

    # --------------------------------------------------------
    # Optional Tree core membership
    # --------------------------------------------------------
    tree_core_set = set()

    if (
        tree_core_subset_name is not None
        and tree_core_subset_name
        in tree_subsets
    ):
        tree_core_set = set(
            tree_subsets[
                tree_core_subset_name
            ]
        )

    # --------------------------------------------------------
    # Start from features that Core excluded.
    # --------------------------------------------------------
    rescue_base = (
        selection_table
        .loc[
            ~selection_table[
                "feature"
            ].isin(
                core_set
            )
        ]
        .drop_duplicates(
            subset=["feature"]
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    if rescue_base.empty:
        return {
            "core_subset": (
                lr_core_subset_name
            ),
            "core_features": (
                core_features
            ),
            "rescue_evidence": (
                pd.DataFrame()
            ),
            "rescue_summary": (
                pd.DataFrame()
            ),
            "rescue_features": [],
            "fe_source_pool": (
                core_features
            ),
        }

    # --------------------------------------------------------
    # Route-rank diagnostics
    # --------------------------------------------------------
    lr_rank_table = (
        _route_rank_table(
            lr_routes,
            rescue_base[
                "feature"
            ].tolist(),
            prefix="lr",
            front_k=k,
        )
    )

    tree_rank_table = (
        _route_rank_table(
            tree_routes,
            rescue_base[
                "feature"
            ].tolist(),
            prefix="tree",
        )
    )

    rescue_base = (
        rescue_base
        .merge(
            lr_rank_table,
            on="feature",
            how="left",
        )
        .merge(
            tree_rank_table,
            on="feature",
            how="left",
        )
    )

    # --------------------------------------------------------
    # Atomic evidence used by rescue channels
    # --------------------------------------------------------
    rescue_base[
        "iv_percentile"
    ] = _safe_numeric_column(
        rescue_base,
        "iv_percentile",
    )

    rescue_base[
        "mi_percentile"
    ] = _safe_numeric_column(
        rescue_base,
        "mi_percentile",
    )

    rescue_base[
        "gain_percentile"
    ] = _safe_numeric_column(
        rescue_base,
        "gain_percentile",
    )

    rescue_base[
        "permutation_percentile"
    ] = _safe_numeric_column(
        rescue_base,
        "permutation_percentile",
    )

    rescue_base[
        "nonlinear_gap_mi_minus_iv"
    ] = (
        rescue_base[
            "mi_percentile"
        ]
        - rescue_base[
            "iv_percentile"
        ]
    )

    rescue_base[
        "tree_model_signal"
    ] = (
        rescue_base[
            [
                "gain_percentile",
                "permutation_percentile",
            ]
        ]
        .max(
            axis=1,
            skipna=True,
        )
    )

    rescue_base[
        "tree_vs_lr_rank_gap"
    ] = (
        rescue_base[
            "lr_best_rank"
        ]
        - rescue_base[
            "tree_best_rank"
        ]
    )

    rescue_base[
        "tree_core_member"
    ] = (
        rescue_base[
            "feature"
        ]
        .isin(
            tree_core_set
        )
    )

    exclusive_flag = (
        _safe_bool_column(
            rescue_base,
            "interaction_exclusive_rescue_candidate",
        )
    )

    general_flag = (
        _safe_bool_column(
            rescue_base,
            "interaction_rescue_candidate",
        )
    )

    rescue_base[
        "interaction_rescue_flag"
    ] = (
        exclusive_flag
        | general_flag
    )

    interaction_strength_candidates = []

    for column in [
        "interaction_exclusive_max_shared_path_count",
        "max_shared_path_count",
    ]:
        if column in rescue_base.columns:
            interaction_strength_candidates.append(
                _safe_numeric_column(
                    rescue_base,
                    column,
                )
            )

    if interaction_strength_candidates:
        rescue_base[
            "interaction_strength"
        ] = (
            pd.concat(
                interaction_strength_candidates,
                axis=1,
            )
            .max(
                axis=1,
                skipna=True,
            )
        )
    else:
        rescue_base[
            "interaction_strength"
        ] = np.nan

    # --------------------------------------------------------
    # Channel A — nonlinear disagreement
    # --------------------------------------------------------
    nonlinear_mask = (
        rescue_base[
            "mi_percentile"
        ].ge(
            LR_RESCUE_MI_MIN
        )
        & (
            rescue_base[
                "iv_percentile"
            ].isna()
            |
            rescue_base[
                "nonlinear_gap_mi_minus_iv"
            ].ge(
                LR_RESCUE_MI_IV_GAP_MIN
            )
        )
    )

    nonlinear = _take_channel(
        rescue_base.loc[
            nonlinear_mask
        ].copy(),
        channel="nonlinear_disagreement",
        budget=(
            LR_RESCUE_NONLINEAR_BUDGET
        ),
        sort_columns=[
            "nonlinear_gap_mi_minus_iv",
            "mi_percentile",
            "tree_model_signal",
            "lr_best_rank",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
        rationale=(
            "MI is strong relative to IV/current LR evidence; "
            "reserve for log/bin/spline-style representation search."
        ),
    )

    # --------------------------------------------------------
    # Channel B — Tree–LR disagreement
    # --------------------------------------------------------
    tree_front_limit = min(
        len(
            candidate_features
        ),
        max(
            k
            + LR_RESCUE_TREE_FRONT_EXTRA,
            300,
        ),
    )

    tree_disagreement_mask = (
        rescue_base[
            "tree_core_member"
        ]
        |
        (
            rescue_base[
                "tree_best_rank"
            ].le(
                tree_front_limit
            )
            & (
                rescue_base[
                    "tree_vs_lr_rank_gap"
                ].ge(
                    LR_RESCUE_TREE_RANK_GAP_MIN
                )
                |
                rescue_base[
                    "tree_model_signal"
                ].ge(
                    LR_RESCUE_TREE_MODEL_MIN
                )
            )
        )
    )

    tree_disagreement = _take_channel(
        rescue_base.loc[
            tree_disagreement_mask
        ].copy(),
        channel="tree_lr_disagreement",
        budget=(
            LR_RESCUE_TREE_DISAGREEMENT_BUDGET
        ),
        sort_columns=[
            "tree_core_member",
            "tree_vs_lr_rank_gap",
            "tree_model_signal",
            "tree_best_rank",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
        rationale=(
            "Tree evidence/ranking is materially stronger than LR ranking; "
            "reserve as a threshold/nonlinearity/interaction clue."
        ),
    )

    # --------------------------------------------------------
    # Channel C — interaction evidence
    # --------------------------------------------------------
    interaction_mask = (
        rescue_base[
            "interaction_rescue_flag"
        ]
        |
        rescue_base[
            "interaction_strength"
        ].fillna(
            0.0
        ).gt(
            0.0
        )
    )

    interaction = _take_channel(
        rescue_base.loc[
            interaction_mask
        ].copy(),
        channel="interaction_evidence",
        budget=(
            LR_RESCUE_INTERACTION_BUDGET
        ),
        sort_columns=[
            "interaction_rescue_flag",
            "interaction_strength",
            "tree_best_rank",
            "mi_percentile",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
        rationale=(
            "NB8 tree-path evidence suggests joint usefulness; "
            "reserve for restricted LR interaction construction."
        ),
    )

    # --------------------------------------------------------
    # Channel D — boundary disagreement across LR routes
    # --------------------------------------------------------
    boundary_limit = min(
        len(
            candidate_features
        ),
        k
        + LR_RESCUE_BOUNDARY_MARGIN,
    )

    boundary_mask = (
        rescue_base[
            "lr_best_rank"
        ].le(
            boundary_limit
        )
    )

    boundary = _take_channel(
        rescue_base.loc[
            boundary_mask
        ].copy(),
        channel="lr_boundary_disagreement",
        budget=(
            LR_RESCUE_BOUNDARY_BUDGET
        ),
        sort_columns=[
            "lr_front_votes_at_k",
            "lr_best_rank",
            "lr_median_rank",
            "mi_percentile",
        ],
        ascending=[
            False,
            True,
            True,
            False,
        ],
        rationale=(
            "Excluded by the final Core membership but remains near the "
            "competitive boundary in one or more LR routes."
        ),
    )

    channel_frames = [
        frame
        for frame in [
            nonlinear,
            tree_disagreement,
            interaction,
            boundary,
        ]
        if not frame.empty
    ]

    if channel_frames:
        rescue_evidence = pd.concat(
            channel_frames,
            ignore_index=True,
        )
    else:
        rescue_evidence = (
            pd.DataFrame()
        )

    if rescue_evidence.empty:
        return {
            "core_subset": (
                lr_core_subset_name
            ),
            "core_features": (
                core_features
            ),
            "rescue_evidence": (
                rescue_evidence
            ),
            "rescue_summary": (
                pd.DataFrame()
            ),
            "rescue_features": [],
            "fe_source_pool": (
                core_features
            ),
        }

    channel_priority = {
        "interaction_evidence": 1,
        "nonlinear_disagreement": 2,
        "tree_lr_disagreement": 3,
        "lr_boundary_disagreement": 4,
    }

    rescue_evidence[
        "channel_priority"
    ] = (
        rescue_evidence[
            "rescue_channel"
        ]
        .map(
            channel_priority
        )
    )

    # A feature supported by multiple rescue channels receives
    # higher priority in the final union, but we keep every
    # channel row for auditability.
    rescue_summary = (
        rescue_evidence
        .groupby(
            "feature",
            as_index=False,
        )
        .agg(
            rescue_channel_count=(
                "rescue_channel",
                "nunique",
            ),
            rescue_channels=(
                "rescue_channel",
                lambda values: "|".join(
                    sorted(
                        set(values)
                    )
                ),
            ),
            best_channel_priority=(
                "channel_priority",
                "min",
            ),
            best_channel_rank=(
                "channel_rank",
                "min",
            ),
            mi_percentile=(
                "mi_percentile",
                "max",
            ),
            iv_percentile=(
                "iv_percentile",
                "max",
            ),
            tree_model_signal=(
                "tree_model_signal",
                "max",
            ),
            tree_vs_lr_rank_gap=(
                "tree_vs_lr_rank_gap",
                "max",
            ),
            interaction_strength=(
                "interaction_strength",
                "max",
            ),
        )
        .sort_values(
            [
                "rescue_channel_count",
                "best_channel_priority",
                "best_channel_rank",
                "feature",
            ],
            ascending=[
                False,
                True,
                True,
                True,
            ],
        )
        .head(
            LR_RESCUE_MAX_TOTAL
        )
        .reset_index(
            drop=True
        )
    )

    rescue_features = (
        rescue_summary[
            "feature"
        ]
        .tolist()
    )

    fe_source_pool = list(
        dict.fromkeys(
            [
                *core_features,
                *rescue_features,
            ]
        )
    )

    return {
        "core_subset": (
            lr_core_subset_name
        ),
        "core_features": (
            core_features
        ),
        "rescue_evidence": (
            rescue_evidence
        ),
        "rescue_summary": (
            rescue_summary
        ),
        "rescue_features": (
            rescue_features
        ),
        "fe_source_pool": (
            fe_source_pool
        ),
    }


# ------------------------------------------------------------
# Preview based on the current 1-SE-style recommendation.
# Final freeze remains manual in the next cell.
# ------------------------------------------------------------
lr_rescue_preview = None

preview_lr_core_subset = (
    lr_recommendation[
        "subset"
    ]
    if lr_recommendation is not None
    else None
)

preview_tree_core_subset = (
    tree_recommendation[
        "subset"
    ]
    if tree_recommendation is not None
    else None
)


if preview_lr_core_subset is not None:

    lr_rescue_preview = (
        build_lr_source_rescue(
            lr_core_subset_name=(
                preview_lr_core_subset
            ),
            tree_core_subset_name=(
                preview_tree_core_subset
            ),
        )
    )

    print(
        "LR Core source features:",
        len(
            lr_rescue_preview[
                "core_features"
            ]
        ),
    )

    print(
        "LR Rescue features:",
        len(
            lr_rescue_preview[
                "rescue_features"
            ]
        ),
    )

    print(
        "LR NB11 source pool:",
        len(
            lr_rescue_preview[
                "fe_source_pool"
            ]
        ),
    )

    print(
        "Rescue channel counts:"
    )

    display(
        lr_rescue_preview[
            "rescue_evidence"
        ][
            [
                "rescue_channel",
                "feature",
            ]
        ]
        .groupby(
            "rescue_channel"
        )
        .size()
        .rename(
            "candidate_rows"
        )
        .reset_index()
    )

    print(
        "Top LR rescue features:"
    )

    display(
        lr_rescue_preview[
            "rescue_summary"
        ]
        .head(
            30
        )
    )

else:
    print(
        "LR rescue preview 尚未生成："
        "先运行 Stage 4 F1–F4，得到 lr_recommendation。"
    )


In [33]:
# ============================================================
# Save results + upstream-definition freeze guardrails
# ============================================================

def save_if_not_empty(
    frame,
    filename,
):
    if (
        isinstance(
            frame,
            pd.DataFrame,
        )
        and not frame.empty
    ):
        frame.to_csv(
            REPORT_ROOT
            / filename,
            index=False,
        )


save_if_not_empty(
    tree_coarse_ranked,
    "tree_coarse_results.csv",
)

save_if_not_empty(
    lr_coarse_ranked,
    "lr_coarse_results.csv",
)

save_if_not_empty(
    tree_boundary_ranked,
    "tree_boundary_results.csv",
)

save_if_not_empty(
    lr_boundary_ranked,
    "lr_boundary_results.csv",
)

save_if_not_empty(
    tree_fine_ranked,
    "tree_fine_results.csv",
)

save_if_not_empty(
    lr_fine_ranked,
    "lr_fine_results.csv",
)

save_if_not_empty(
    tree_relevance_variant_ranked,
    "tree_relevance_variant_followup.csv",
)

save_if_not_empty(
    lr_relevance_variant_ranked,
    "lr_relevance_variant_followup.csv",
)

save_if_not_empty(
    tree_repair_ranked,
    "tree_local_repair_results.csv",
)

save_if_not_empty(
    lr_repair_ranked,
    "lr_local_repair_results.csv",
)

save_if_not_empty(
    tree_repair_catalog,
    "tree_local_repair_catalog.csv",
)

save_if_not_empty(
    lr_repair_catalog,
    "lr_local_repair_catalog.csv",
)

save_if_not_empty(
    tree_search_final,
    "tree_search_final_before_f1_f4.csv",
)

save_if_not_empty(
    lr_search_final,
    "lr_search_final_before_f1_f4.csv",
)

save_if_not_empty(
    tree_final_ranked,
    "tree_full_temporal_results.csv",
)

save_if_not_empty(
    lr_final_ranked,
    "lr_full_temporal_results.csv",
)

save_if_not_empty(
    tree_fold_metrics,
    "tree_f1_f4.csv",
)

save_if_not_empty(
    lr_fold_metrics,
    "lr_f1_f4.csv",
)

save_if_not_empty(
    nb9_preflight,
    "nb9_to_nb10_preflight.csv",
)


# LR source-rescue preview is advisory until manual freeze.
if lr_rescue_preview is not None:

    save_if_not_empty(
        lr_rescue_preview[
            "rescue_evidence"
        ],
        "lr_source_rescue_preview_evidence.csv",
    )

    save_if_not_empty(
        lr_rescue_preview[
            "rescue_summary"
        ],
        "lr_source_rescue_preview_summary.csv",
    )

    with open(
        REPORT_ROOT
        / "lr_source_rescue_preview.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            {
                "core_subset": (
                    lr_rescue_preview[
                        "core_subset"
                    ]
                ),
                "core_feature_count": len(
                    lr_rescue_preview[
                        "core_features"
                    ]
                ),
                "rescue_feature_count": len(
                    lr_rescue_preview[
                        "rescue_features"
                    ]
                ),
                "fe_source_pool_count": len(
                    lr_rescue_preview[
                        "fe_source_pool"
                    ]
                ),
                "rescue_features": (
                    lr_rescue_preview[
                        "rescue_features"
                    ]
                ),
            },
            file,
            ensure_ascii=False,
            indent=2,
        )


def recommendation_guardrail(
    recommendation,
    branch,
):
    """
    不是重新评估 candidate。
    只是检查最终推荐是否踩中了 NB9 已知的定义敏感点。
    """
    if recommendation is None:
        return {
            "branch": branch,
            "subset": None,
            "route": None,
            "freeze_status": (
                "NO_RECOMMENDATION_YET"
            ),
            "required_followup": (
                "Run full temporal evaluation first."
            ),
        }

    route = str(
        recommendation.get(
            "route",
            "unknown",
        )
    )

    followups = []


    if (
        route
        .startswith(
            "interaction_relevance"
        )
        and NB9_GUARDRAILS.get(
            "interaction_pair_limit_sensitive",
            False,
        )
    ):
        followups.append(
            "NB9 Interaction route-definition follow-up: "
            "current 300-pair formal route was sensitive "
            "to wider pair limits."
        )


    if (
        route
        .startswith(
            "temporal_relevance"
        )
        and NB9_GUARDRAILS.get(
            "temporal_boundary_sensitive",
            False,
        )
    ):
        followups.append(
            "NB9 Temporal route-definition follow-up: "
            "many features lie near the current min-Gini boundary."
        )


    if (
        route == "relevance"
        and NB9_GUARDRAILS.get(
            "relevance_local_variant_material",
            False,
        )
        and not RUN_RELEVANCE_VARIANT_FOLLOWUP
    ):
        followups.append(
            "Run the NB9-saved nearby Relevance variant follow-up "
            "at the competitive depth before freeze."
        )


    # Consensus inherits the definitions of its component routes.
    # If those upstream definitions are known to be sensitive,
    # a winning Consensus should be rebuilt / rechecked in NB9
    # before the final feature list is frozen.
    if route in {
        "consensus",
        "lr_consensus",
    }:

        consensus_sensitive = []

        if NB9_GUARDRAILS.get(
            "relevance_local_variant_material",
            False,
        ):
            consensus_sensitive.append(
                "Relevance weighting"
            )

        if NB9_GUARDRAILS.get(
            "temporal_boundary_sensitive",
            False,
        ):
            consensus_sensitive.append(
                "Temporal boundary"
            )

        if (
            branch == "tree"
            and NB9_GUARDRAILS.get(
                "interaction_pair_limit_sensitive",
                False,
            )
        ):
            consensus_sensitive.append(
                "Interaction pair-limit"
            )

        if consensus_sensitive:
            followups.append(
                "NB9 Consensus robustness follow-up for: "
                + ", ".join(
                    consensus_sensitive
                )
                + "."
            )


    freeze_status = (
        "FOLLOWUP_REQUIRED"
        if followups
        else "READY_FOR_MANUAL_FREEZE_REVIEW"
    )


    return {
        "branch": branch,
        "subset": recommendation.get(
            "subset"
        ),
        "route": route,
        "freeze_status": (
            freeze_status
        ),
        "required_followup": (
            " | ".join(
                followups
            )
            if followups
            else (
                "No NB9 definition-sensitive "
                "guardrail triggered."
            )
        ),
    }


tree_freeze_guardrail = (
    recommendation_guardrail(
        tree_recommendation,
        "tree",
    )
)

lr_freeze_guardrail = (
    recommendation_guardrail(
        lr_recommendation,
        "lr",
    )
)


freeze_guardrail_table = (
    pd.DataFrame([
        tree_freeze_guardrail,
        lr_freeze_guardrail,
    ])
)


print(
    "Freeze guardrails"
)

display(
    freeze_guardrail_table
)


recommendations = {
    "tree": (
        tree_recommendation
    ),
    "lr": (
        lr_recommendation
    ),
    "tree_freeze_guardrail": (
        tree_freeze_guardrail
    ),
    "lr_freeze_guardrail": (
        lr_freeze_guardrail
    ),
    "nb9_guardrails": (
        NB9_GUARDRAILS
    ),
    "validation_read": False,
    "oot_read": False,
    "coarse_sizes": (
        COARSE_SIZES
    ),
    "fine_step": (
        FINE_STEP
    ),
    "boundary_expansion_step": (
        BOUNDARY_EXPANSION_STEP
    ),
    "max_boundary_rounds": (
        MAX_BOUNDARY_ROUNDS
    ),
    "local_repair_plan": (
        local_repair_plan
    ),
    "checkpoint_root": str(
        CHECKPOINT_ROOT
    ),
    "device_type": (
        DEVICE_TYPE
    ),
    "lgbm_max_bin": (
        LGBM_MAX_BIN
    ),
    "nb9_tree_routes": list(
        tree_routes
    ),
    "nb9_lr_routes": list(
        lr_routes
    ),

    "lr_nb10_output_role": (
        "core source set + transformation rescue pool; "
        "final LR design matrix is NOT frozen in NB10"
    ),

    "lr_rescue_config": {
        "nonlinear_budget": (
            LR_RESCUE_NONLINEAR_BUDGET
        ),
        "tree_disagreement_budget": (
            LR_RESCUE_TREE_DISAGREEMENT_BUDGET
        ),
        "interaction_budget": (
            LR_RESCUE_INTERACTION_BUDGET
        ),
        "boundary_budget": (
            LR_RESCUE_BOUNDARY_BUDGET
        ),
        "max_total_rescue": (
            LR_RESCUE_MAX_TOTAL
        ),
        "boundary_margin": (
            LR_RESCUE_BOUNDARY_MARGIN
        ),
        "validation_read": False,
        "oot_read": False,
    },
}


with open(
    REPORT_ROOT
    / "nb10_recommendations.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        recommendations,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# Freeze remains manual.
# ------------------------------------------------------------
#
# Tree:
#   NB10 can freeze the actual source feature list.
#
# LR:
#   NB10 freezes only:
#   1) Core source features
#   2) Transformation Rescue Pool
#   3) Their union = NB11 FE source pool
#
# Final transformed LR design matrix will be frozen in NB11.

FREEZE_FINAL_FEATURES = False

FINAL_TREE_SUBSET = None
FINAL_LR_CORE_SUBSET = None


if FREEZE_FINAL_FEATURES:

    # --------------------------------------------------------
    # Tree final freeze
    # --------------------------------------------------------
    if FINAL_TREE_SUBSET:

        if (
            tree_freeze_guardrail[
                "freeze_status"
            ]
            != "READY_FOR_MANUAL_FREEZE_REVIEW"
        ):
            raise RuntimeError(
                "Tree freeze blocked by NB9/NB10 robustness guardrail. "
                "Resolve the required follow-up first."
            )

        if (
            FINAL_TREE_SUBSET
            not in tree_subsets
        ):
            raise KeyError(
                f"Unknown FINAL_TREE_SUBSET: {FINAL_TREE_SUBSET}"
            )

        with open(
            REPORT_ROOT
            / "selected_features_tree.json",
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                tree_subsets[
                    FINAL_TREE_SUBSET
                ],
                file,
                ensure_ascii=False,
                indent=2,
            )


    # --------------------------------------------------------
    # LR source-space freeze for NB11
    # --------------------------------------------------------
    if FINAL_LR_CORE_SUBSET:

        if (
            lr_freeze_guardrail[
                "freeze_status"
            ]
            != "READY_FOR_MANUAL_FREEZE_REVIEW"
        ):
            raise RuntimeError(
                "LR source-space freeze blocked by "
                "NB9/NB10 robustness guardrail. "
                "Resolve the required follow-up first."
            )

        if (
            FINAL_LR_CORE_SUBSET
            not in lr_subsets
        ):
            raise KeyError(
                "Unknown FINAL_LR_CORE_SUBSET: "
                f"{FINAL_LR_CORE_SUBSET}"
            )

        lr_frozen_package = (
            build_lr_source_rescue(
                lr_core_subset_name=(
                    FINAL_LR_CORE_SUBSET
                ),
                tree_core_subset_name=(
                    FINAL_TREE_SUBSET
                    if FINAL_TREE_SUBSET
                    in tree_subsets
                    else (
                        tree_recommendation[
                            "subset"
                        ]
                        if (
                            tree_recommendation
                            is not None
                            and tree_recommendation[
                                "subset"
                            ]
                            in tree_subsets
                        )
                        else None
                    )
                ),
            )
        )

        with open(
            REPORT_ROOT
            / "lr_core_source_features.json",
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                lr_frozen_package[
                    "core_features"
                ],
                file,
                ensure_ascii=False,
                indent=2,
            )

        with open(
            REPORT_ROOT
            / "lr_transformation_rescue_features.json",
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                lr_frozen_package[
                    "rescue_features"
                ],
                file,
                ensure_ascii=False,
                indent=2,
            )

        with open(
            REPORT_ROOT
            / "lr_fe_source_pool.json",
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                lr_frozen_package[
                    "fe_source_pool"
                ],
                file,
                ensure_ascii=False,
                indent=2,
            )

        save_if_not_empty(
            lr_frozen_package[
                "rescue_evidence"
            ],
            "lr_transformation_rescue_evidence.csv",
        )

        save_if_not_empty(
            lr_frozen_package[
                "rescue_summary"
            ],
            "lr_transformation_rescue_summary.csv",
        )

        with open(
            REPORT_ROOT
            / "lr_nb11_handoff.json",
            "w",
            encoding="utf-8",
        ) as file:

            json.dump(
                {
                    "lr_core_subset": (
                        FINAL_LR_CORE_SUBSET
                    ),
                    "core_feature_count": len(
                        lr_frozen_package[
                            "core_features"
                        ]
                    ),
                    "rescue_feature_count": len(
                        lr_frozen_package[
                            "rescue_features"
                        ]
                    ),
                    "fe_source_pool_count": len(
                        lr_frozen_package[
                            "fe_source_pool"
                        ]
                    ),
                    "role": (
                        "NB11 source-space handoff; "
                        "not final transformed LR features"
                    ),
                    "validation_read": False,
                    "oot_read": False,
                },
                file,
                ensure_ascii=False,
                indent=2,
            )


print(
    "Notebook 10 outputs:",
    REPORT_ROOT,
)


Freeze guardrails


,branch,subset,route,freeze_status,required_followup
0,tree,tree__relevance__n500,relevance,READY_FOR_MANUAL_FREEZE_REVIEW,No NB9 definition-sensitive guardrail triggered.
1,lr,lr__redundancy_aware__boundary_n100,redundancy_aware,READY_FOR_MANUAL_FREEZE_REVIEW,No NB9 definition-sensitive guardrail triggered.


Notebook 10 outputs: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/final


## 14. 方法总结与下游交接

Notebook 10 使用模型评估确定排序路线的搜索深度，并通过 boundary / fine search、restricted local repair 和 F1–F4 时间折诊断检查候选子集。在 Final Pareto 与 plateau 比较之外，还需考虑 Notebook 09 标记的 ranking-definition sensitivity。

| 分支 | 本阶段输出含义 | 后续工作 |
|---|---|---|
| Tree | 条件满足时可冻结的 source feature subset | 模型参数搜索与独立时间验证 |
| Logistic Regression | Core Source Set + Transformation Rescue Pool | NB11 进行分箱、非线性/交互表示搜索及最终筛选 |

**范围限制：** 本 Notebook 不读取 Validation（Week 64–77）或 OOT（Week 78–91）。Train 内时间折参与了特征搜索，因此 F1–F4 结果不应被表述为未参与选择的独立泛化指标。冻结检查未通过的路线应保留待复核状态，不能仅凭局部最优指标视为最终定稿。

```text
NB08 Evidence → NB09 Ranking → NB10 Subset Search
                           ├── Tree: source subset
                           └── LR: core + rescue sources → NB11 representation search
```
